# 03 — Serie temporal: demanda por sabor de cápsula

Tercera y última página de serie temporal. Las dos anteriores trabajaban sobre un único agregado
—suscriptores, euros—; ésta baja al **SKU**, y ahí aparecen dos problemas que el agregado no tiene
y que `docs/data_imperfections.md` documenta como imperfecciones deliberadas:

1. **Demanda censurada por rotura de stock.** En `fct_shop_orders` hay ventanas en las que un sabor
   marca cero ventas, no porque nadie lo quisiera sino porque no había producto. Un cero de demanda
   y un cero de oferta son el mismo número en la tabla y dos cosas opuestas en el negocio.
2. **SKUs descatalogados y relanzados.** Cuando cambia el packaging o el precio, el sabor muere como
   código y nace como otro. Si la serie se agrega por `product_sku`, el sabor tiene dos series cortas
   en vez de una larga.

Las dos se tratan aquí de forma explícita, y en las dos el criterio es el mismo: **no basta con
corregir, hay que demostrar primero que la anomalía existe** — con un contraste, no con un vistazo.
El resto de la página es el tratamiento habitual (descomposición, estacionariedad, backtesting,
forecast), ya con la serie limpia.

Una nota de alcance heredada de la página anterior: los notebooks consumen marts, no tablas `raw_*`.
Aquí son `fct_shipments` (envíos del club) y `fct_shop_orders` (compras puntuales), que son las dos
mitades de la demanda de cápsulas y, como se verá, no se comportan igual ante una rotura de stock.

In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import poisson

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

import utils_timeseries as ts

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"
OUTPUT_PATH = PROJECT_ROOT / "analysis" / "outputs" / "capsulas.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Parámetros del análisis ---
HISTORY_START, HISTORY_END = "2023-09-01", "2026-08-31"
FORECAST_HORIZON = 6
BACKTEST_FOLDS = 5
SEASON_LENGTH = 12
ALPHA = 0.05
INTERVAL_LEVEL = 0.80
SARIMA_ORDER = (0, 1, 1)
SARIMA_SEASONAL = (0, 1, 1, SEASON_LENGTH)

# --- Parámetros de la detección de rotura de stock (se justifican en la sección 3) ---
MIN_RUN_DAYS = 5        # racha mínima de ceros que merece un contraste
REF_DAYS = 45           # ventana de referencia a cada lado para estimar la tasa esperada
MIN_EXPECTED_LINES = 20 # materialidad: por debajo de esto no es un problema de negocio
STOCKOUT_ALPHA = 0.01   # nivel del contraste, ya corregido por multiplicidad

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"
FLAVOR_COLOR = {
    "classic_espresso": "#2a78d6", "origin_colombia": "#eb6834", "intense_10": "#1baf7a",
    "intense_8": "#eda100", "origin_brazil": "#4a3aa7", "decaf_classic": "#e34948",
    "decaf_vanilla": "#8c6d3f", "classic_lungo": "#0f8a8a", "seasonal_pumpkin": "#d4571f",
    "origin_ethiopia": "#7a7a78", "intense_12": "#9c27b0", "seasonal_gingerbread": "#b5893b",
}
FLAVOR_LABEL = {
    "classic_espresso": "Espresso clásico", "classic_lungo": "Lungo clásico",
    "decaf_classic": "Descafeinado", "decaf_vanilla": "Descaf. vainilla",
    "intense_8": "Intenso 8", "intense_10": "Intenso 10", "intense_12": "Intenso 12",
    "origin_brazil": "Origen Brasil", "origin_colombia": "Origen Colombia",
    "origin_ethiopia": "Origen Etiopía", "seasonal_pumpkin": "Temporada calabaza",
    "seasonal_gingerbread": "Temporada jengibre",
}
DAYS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
MONTHS = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=70, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="x unified",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("proyecto:", PROJECT_ROOT.name, "| duckdb:", DB_PATH.exists())

proyecto: capsule-club-analytics | duckdb: True


## 1. Las dos mitades de la demanda, y el eje correcto del SKU

La demanda de un sabor se reparte entre dos marts que no se comportan igual:

| Mart | Qué recoge | Ante una rotura de stock |
|---|---|---|
| `fct_shipments` | Envíos del club a suscriptores. | No se pierde: el catálogo del envío se decide en almacén y la demanda se sirve igual. |
| `fct_shop_orders` | Compras puntuales, online y boutique. | **Se censura**: la línea de pedido desaparece y en la tabla parece demanda cero. |

Que la imperfección afecte sólo a una de las dos mitades no es un detalle: convierte al club en el
**grupo de control natural** de la tienda. Ese es el eje de toda la sección 3.

Sobre el grano: los dos marts traen ya `canonical_sku`, que resuelve la cadena `replaced_by_sku` en
`int_product_sku_continuity`. La serie se agrega por **sabor** (equivalente a `canonical_sku` en
cápsulas), no por `product_sku`, y la sección 2 enseña qué pasa si no se hace.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

shop = con.sql("""
    select order_date            as d,
           flavor,
           product_sku,
           canonical_sku,
           sum(quantity)         as units,
           count(*)              as lines,
           sum(line_amount_eur)  as eur
    from fct_shop_orders
    where is_capsule
    group by 1, 2, 3, 4
""").df()

club = con.sql("""
    select ship_date                as d,
           flavor,
           capsule_sku              as product_sku,
           canonical_sku,
           sum(quantity)            as units,
           count(*)                 as lines,
           sum(shipment_value_eur)  as eur
    from fct_shipments
    group by 1, 2, 3, 4
""").df()

catalog = con.sql("""
    select * from int_product_sku_continuity where product_type = 'capsule'
""").df()

shop["d"] = pd.to_datetime(shop.d)
club["d"] = pd.to_datetime(club.d)
FLAVORS = sorted(catalog.flavor.unique())

summary = pd.DataFrame({
    "unidades": {"tienda": shop.units.sum(), "club": club.units.sum()},
    "lineas": {"tienda": shop.lines.sum(), "club": club.lines.sum()},
    "euros": {"tienda": shop.eur.sum(), "club": club.eur.sum()},
})
summary["% unidades"] = summary.unidades / summary.unidades.sum() * 100
print(summary.round(1).to_string())
print()
print(f"{len(FLAVORS)} sabores · {catalog.product_sku.nunique()} SKUs de cápsula "
      f"({int(catalog.is_relaunch_version.sum())} de ellos son relanzamientos)")

        unidades  lineas      euros  % unidades
tienda  251735.0   66865  1265665.8        68.9
club    113715.0   45508   575659.5        31.1

12 sabores · 15 SKUs de cápsula (3 de ellos son relanzamientos)


La tienda es el **69% de las unidades** y el club el 31%. Conviene dejarlo escrito porque
condiciona cómo se lee toda la página: **la mitad censurable del negocio es la mayor**. La cabecera
de `fct_shipments` decía lo contrario ("el club es el grueso del volumen"); era deuda de
documentación del mart, no un problema de datos, y se ha corregido al construir esta página.

A partir de aquí todo se trabaja sobre rejillas diarias `fecha × sabor`, reindexadas contra el
calendario completo. El reindexado importa: un día sin filas no puede desaparecer de la serie,
porque ese cero es justo la señal que hay que interpretar.

In [3]:
CALENDAR = pd.date_range(HISTORY_START, HISTORY_END, freq="D")

def grid(frame, value_col):
    """Rejilla diaria fecha x sabor, sin huecos de calendario."""
    return (frame.pivot_table(index="d", columns="flavor", values=value_col, aggfunc="sum")
                 .reindex(CALENDAR)
                 .reindex(columns=FLAVORS)
                 .fillna(0.0))

shop_units, shop_lines, shop_eur = grid(shop, "units"), grid(shop, "lines"), grid(shop, "eur")
club_units, club_eur = grid(club, "units"), grid(club, "eur")

# Vida de catálogo: un sabor sólo puede vender mientras alguna de sus versiones está viva.
# Es un invariante garantizado del generador, así que fuera de esta máscara el cero no es
# información: es ausencia de producto conocida y no hay nada que contrastar.
alive = pd.DataFrame(False, index=CALENDAR, columns=FLAVORS)
for row in catalog.itertuples():
    launch = pd.Timestamp(row.launch_date)
    end = pd.Timestamp(row.discontinue_date) if pd.notna(row.discontinue_date) else CALENDAR[-1]
    alive.loc[max(launch, CALENDAR[0]):min(end, CALENDAR[-1]), row.flavor] = True

coverage = pd.DataFrame({
    "dias_en_catalogo": alive.sum(),
    "dias_con_venta_tienda": (shop_lines > 0).sum(),
    "unidades_tienda": shop_units.sum(),
    "unidades_club": club_units.sum(),
})
coverage["dias_a_cero_en_catalogo"] = coverage.dias_en_catalogo - coverage.dias_con_venta_tienda
print(coverage.sort_values("unidades_tienda", ascending=False).round(0).to_string())

                      dias_en_catalogo  dias_con_venta_tienda  unidades_tienda  unidades_club  dias_a_cero_en_catalogo
classic_espresso                  1096                   1043          40223.0        15727.0                       53
origin_colombia                   1096                    995          28584.0        19558.0                      101
intense_10                        1096                    995          28115.0         9860.0                      101
intense_8                         1096                   1010          25903.0         9795.0                       86
decaf_classic                     1096                    983          24368.0         7381.0                      113
decaf_vanilla                     1096                    954          22673.0         6821.0                      142
seasonal_pumpkin                   898                    736          20599.0         1926.0                      162
origin_brazil                     1096          

## 2. La discontinuidad de los SKUs relanzados

Tres de los doce sabores cambian de código a mitad del histórico. Visto por `product_sku`, cada uno
de ellos es una serie que muere y otra que nace el mismo mes.

In [4]:
relaunched = (catalog[catalog.flavor_was_relaunched]
              .sort_values(["flavor", "version_number"]))
switch_dates = {row.flavor: pd.Timestamp(row.launch_date)
                for row in relaunched.itertuples() if row.is_relaunch_version}

sku_monthly = (pd.concat([shop[["d", "flavor", "product_sku", "units", "eur"]],
                          club[["d", "flavor", "product_sku", "units", "eur"]]])
                 .groupby(["flavor", "product_sku", pd.Grouper(key="d", freq="MS")])
                 [["units", "eur"]].sum().reset_index())

fig = make_subplots(rows=1, cols=3, subplot_titles=[FLAVOR_LABEL[f] for f in switch_dates],
                    shared_yaxes=False)
for col, flavor in enumerate(switch_dates, start=1):
    sub = sku_monthly[sku_monthly.flavor == flavor]
    for i, sku in enumerate(sorted(sub.product_sku.unique())):
        part = sub[sub.product_sku == sku]
        fig.add_trace(go.Scatter(x=part.d, y=part.units, mode="lines",
                                 name=sku, legendgroup=sku,
                                 line=dict(color=C_BLUE if i == 0 else C_ORANGE, width=2),
                                 showlegend=False), row=1, col=col)
    fig.add_vline(x=switch_dates[flavor], line=dict(color=C_MUTED, width=1, dash="dot"),
                  row=1, col=col)
fig.update_layout(**{**PLOT_LAYOUT, "height": 340, "hovermode": "closest"},
                  title="Demanda mensual por product_sku: V1 (azul) y V2 (naranja)")
fig.update_yaxes(title_text="unidades / mes", col=1)
fig.show()

price_change = []
for flavor, switch in switch_dates.items():
    versions = catalog[catalog.flavor == flavor].sort_values("version_number")
    realised = (shop[shop.flavor == flavor].groupby("product_sku")
                .apply(lambda g: g.eur.sum() / g.units.sum(), include_groups=False))
    price_change.append({
        "sabor": flavor,
        "fecha_cambio": switch.date(),
        "sku_v1": versions.product_sku.iloc[0], "sku_v2": versions.product_sku.iloc[1],
        "precio_lista_v1": versions.list_unit_price_eur.iloc[0],
        "precio_lista_v2": versions.list_unit_price_eur.iloc[1],
        "precio_real_v1": realised.get(versions.product_sku.iloc[0], np.nan),
        "precio_real_v2": realised.get(versions.product_sku.iloc[1], np.nan),
    })
price_change = pd.DataFrame(price_change)
price_change["subida_pct"] = (price_change.precio_real_v2 / price_change.precio_real_v1 - 1) * 100
print(price_change.round(3).to_string(index=False))

               sabor fecha_cambio                      sku_v1                      sku_v2  precio_lista_v1  precio_lista_v2  precio_real_v1  precio_real_v2  subida_pct
    classic_espresso   2025-10-25     CAP-CLASSIC-ESPRESSO-V1     CAP-CLASSIC-ESPRESSO-V2             5.93             6.57           5.714           6.321      10.612
       origin_brazil   2025-10-21        CAP-ORIGIN-BRAZIL-V1        CAP-ORIGIN-BRAZIL-V2             4.85             5.28           4.666           5.083       8.931
seasonal_gingerbread   2025-03-24 CAP-SEASONAL-GINGERBREAD-V1 CAP-SEASONAL-GINGERBREAD-V2             5.63             6.22           5.426           5.978      10.174


El corte es limpio y la continuidad se arregla agregando por `canonical_sku`. Pero antes de darlo por
resuelto conviene separar dos cosas que el relanzamiento mezcla:

- **En unidades**, la discontinuidad es un artefacto de codificación: el sabor no ha desaparecido, ha
  cambiado de etiqueta. Aquí sí hay que unir las dos series.
- **En euros, no.** El relanzamiento sube el precio real entre un 9% y un 11%, así que la serie de
  ingresos de ese sabor tiene un escalón **auténtico** que no hay que suavizar. Unir los SKUs y
  además alisar el salto de precio sería borrar un cambio de negocio real.

Queda una tercera pregunta, que es la que de verdad importa para el forecast: **¿el relanzamiento
cambió el volumen?** Si subir el precio un 10% movió la demanda, la serie unida tiene un cambio de
nivel y el modelo debería saberlo.

In [5]:
# Diferencias en diferencias sobre unidades totales (tienda + club).
#
# El grupo de control son los sabores no relanzados y no estacionales: absorben el crecimiento del
# negocio y el ciclo anual común. Los estacionales se excluyen del control a propósito, porque su
# ciclo propio (calabaza en otoño, jengibre en Navidad) no es comparable con el del resto.
demand_daily_raw = shop_units + club_units
CONTROL = [f for f in FLAVORS if f not in switch_dates and not f.startswith("seasonal_")]
DID_WINDOW, BLOCK_DAYS, N_BOOT = 120, 7, 2000
rng = np.random.default_rng(20240215)

def did_estimate(frame, flavor, switch, window=DID_WINDOW):
    pre = frame.loc[switch - pd.Timedelta(days=window):switch - pd.Timedelta(days=1)]
    post = frame.loc[switch:switch + pd.Timedelta(days=window - 1)]
    if pre[flavor].sum() == 0 or post[flavor].sum() == 0:
        return np.nan
    return (np.log(post[flavor].sum() / pre[flavor].sum())
            - np.log(post[CONTROL].to_numpy().sum() / pre[CONTROL].to_numpy().sum()))

did_rows = []
for flavor, switch in switch_dates.items():
    point = did_estimate(demand_daily_raw, flavor, switch)
    pre = demand_daily_raw.loc[switch - pd.Timedelta(days=DID_WINDOW):switch - pd.Timedelta(days=1)]
    post = demand_daily_raw.loc[switch:switch + pd.Timedelta(days=DID_WINDOW - 1)]

    # (a) error estándar de manual, suponiendo Poisson: cada unidad, independiente.
    se_poisson = np.sqrt(1 / pre[flavor].sum() + 1 / post[flavor].sum()
                         + 1 / pre[CONTROL].to_numpy().sum() + 1 / post[CONTROL].to_numpy().sum())

    # (b) bootstrap por bloques de una semana, que no supone nada sobre la varianza.
    def blocks(frame):
        return [frame.iloc[k * BLOCK_DAYS:(k + 1) * BLOCK_DAYS]
                for k in range(len(frame) // BLOCK_DAYS)]
    b_pre, b_post = blocks(pre), blocks(post)
    draws = []
    for _ in range(N_BOOT):
        P = pd.concat([b_pre[k] for k in rng.integers(0, len(b_pre), len(b_pre))])
        Q = pd.concat([b_post[k] for k in rng.integers(0, len(b_post), len(b_post))])
        if P[flavor].sum() == 0 or Q[flavor].sum() == 0:
            continue
        draws.append(np.log(Q[flavor].sum() / P[flavor].sum())
                     - np.log(Q[CONTROL].to_numpy().sum() / P[CONTROL].to_numpy().sum()))
    boot_lo, boot_hi = np.percentile(draws, [2.5, 97.5])

    # (c) placebo: el mismo estimador en fechas en las que no pasó nada.
    placebos = [did_estimate(demand_daily_raw, flavor, d)
                for d in pd.date_range("2024-02-01", "2026-04-30", freq="14D")
                if abs((d - switch).days) >= DID_WINDOW]
    placebos = np.array([p for p in placebos if p == p])
    p_placebo = float((np.abs(placebos) >= abs(point)).mean())

    did_rows.append({
        "sabor": flavor,
        "efecto_pct": (np.exp(point) - 1) * 100,
        "ic95_poisson": f"[{(np.exp(point - 1.96 * se_poisson) - 1) * 100:+.1f}, "
                        f"{(np.exp(point + 1.96 * se_poisson) - 1) * 100:+.1f}]",
        "ic95_bootstrap": f"[{(np.exp(boot_lo) - 1) * 100:+.1f}, {(np.exp(boot_hi) - 1) * 100:+.1f}]",
        "sd_placebos_pct": float(np.std(np.exp(placebos) - 1) * 100),
        "n_placebos": int(len(placebos)),
        "p_placebo": p_placebo,
    })

did_table = pd.DataFrame(did_rows)
print(did_table.round(3).to_string(index=False))
print()
overdispersion = {f: float(demand_daily_raw[f].loc["2024-09":].var()
                           / demand_daily_raw[f].loc["2024-09":].mean()) for f in switch_dates}
print("razón varianza/media de la demanda diaria (Poisson implicaría 1):")
for f, v in overdispersion.items():
    print(f"  {f:22s} {v:5.1f}")

               sabor  efecto_pct   ic95_poisson ic95_bootstrap  sd_placebos_pct  n_placebos  p_placebo
    classic_espresso      -1.332   [-4.7, +2.1]   [-8.4, +6.5]            8.453          42      0.929
       origin_brazil      -8.800  [-12.5, -4.9]  [-15.6, -2.0]           11.904          42      0.310
seasonal_gingerbread     -81.038 [-82.9, -79.0] [-88.9, -64.3]          323.090          20      0.500

razón varianza/media de la demanda diaria (Poisson implicaría 1):
  classic_espresso        18.4
  origin_brazil           11.3
  seasonal_gingerbread    40.2


**Ningún relanzamiento movió el volumen de forma medible, y el camino hasta esa conclusión es el
interesante.**

Con el error estándar de manual —el que sale de suponer que cada cápsula vendida es un ensayo
independiente— Brasil parece caer un 8,8% de forma significativa, IC95% `[-12,5; -4,9]`. Pero la
demanda diaria está **sobredispersa entre 11 y 18 veces** respecto a un Poisson, porque las unidades
llegan en pedidos y cada suscriptor tiene un sabor favorito estable: las unidades no son
independientes, los pedidos casi. Un bootstrap por bloques semanales, que no supone nada sobre la
varianza, ensancha el intervalo a `[-15,6; -2,0]`.

Y el placebo lo cierra: aplicando el mismo estimador a 42 fechas en las que no pasó nada, la
desviación típica del efecto "estimado" es del 11,9% y el −8,8% de Brasil queda dentro de lo normal
(p = 0,31). En Espresso el efecto es directamente nulo (−1,3%, p = 0,93).

Con **Jengibre el efecto no es identificable, y conviene decirlo en vez de publicar el número**: su
relanzamiento cae el 24 de marzo de 2025, justo cuando termina su temporada, así que el −81% mide el
final de la Navidad y no el cambio de SKU. No hay contrafactual posible porque la V1 sólo vivió una
temporada.

**Conclusión operativa:** se agrega por `canonical_sku`, sin escalón de nivel que corregir en
unidades, y se deja el escalón de precio intacto en euros.

## 3. Demanda censurada por rotura de stock

`docs/data_imperfections.md` avisa de que hay ventanas en las que un sabor aparece con demanda cero
en tienda porque no había stock. El catálogo no dice cuántas ni cuándo: hay que encontrarlas.

El primer instinto —buscar las rachas de ceros más largas— es también el primero que falla.

In [6]:
def zero_runs(zero_mask, alive_mask, min_len):
    """Rachas maximales de ceros dentro de la vida de catálogo, como (i_inicio, i_fin)."""
    flags = (zero_mask & alive_mask).to_numpy()
    runs, i, n = [], 0, len(flags)
    while i < n:
        if flags[i]:
            j = i
            while j + 1 < n and flags[j + 1]:
                j += 1
            if j - i + 1 >= min_len:
                runs.append((i, j))
            i = j + 1
        else:
            i += 1
    return runs

naive_rows = []
for flavor in FLAVORS:
    for i, j in zero_runs(shop_lines[flavor] == 0, alive[flavor], MIN_RUN_DAYS):
        naive_rows.append({"sabor": flavor, "inicio": CALENDAR[i].date(), "fin": CALENDAR[j].date(),
                           "dias": j - i + 1,
                           "lineas_dia_mes_anterior": float(
                               shop_lines[flavor].iloc[max(0, i - 30):i].mean())})
naive_ranking = (pd.DataFrame(naive_rows).sort_values("dias", ascending=False)
                 .head(10).reset_index(drop=True))
print("Ranking ingenuo: las 10 rachas de ceros más largas")
print(naive_ranking.round(2).to_string(index=False))

Ranking ingenuo: las 10 rachas de ceros más largas
           sabor     inicio        fin  dias  lineas_dia_mes_anterior
 origin_colombia 2024-07-06 2024-07-31    26                     2.57
      intense_10 2026-04-24 2026-05-17    24                    17.27
   decaf_vanilla 2025-06-29 2025-07-18    20                     5.83
   decaf_vanilla 2025-03-09 2025-03-26    18                     6.07
seasonal_pumpkin 2026-02-25 2026-03-14    18                     4.57
   decaf_vanilla 2026-06-22 2026-07-08    17                    13.37
 origin_ethiopia 2023-11-07 2023-11-22    16                     0.20
 origin_ethiopia 2023-09-28 2023-10-13    16                     0.26
      intense_12 2023-10-06 2023-10-20    15                     0.23
 origin_ethiopia 2023-09-01 2023-09-13    13                      NaN


El ranking por duración mezcla dos cosas incomparables. Las rachas de Etiopía en noviembre de 2023 o
de Intenso 12 en octubre de 2023 son largas porque **en 2023 el negocio era diminuto**: un sabor que
vende 0,2 líneas al día pasa dos semanas sin vender por puro azar. Y las de Colombia en julio de 2024
o de Intenso 10 en abril de 2026 son largas sobre sabores que venden 2,6 y 17,3 líneas al día.

Quince días a cero no significan lo mismo en los dos casos. Lo que hay que medir no es la longitud de
la racha sino **cuánta demanda debería haber habido dentro de ella**.

In [7]:
# Tasa esperada dentro de la ventana, estimada con dos exposiciones distintas:
#
#   - "cruzada": lo que vendieron los OTROS sabores esos mismos días. Absorbe la tendencia del
#     negocio, el día de la semana y las campañas, que son comunes a todo el catálogo.
#   - "club": lo que el club envió de ESE sabor esos mismos días. El club no se censura, así que
#     es el control que sí comparte la estacionalidad propia del sabor.
#
# La ventana de referencia (+-45 días) se deja corta a propósito: con +-90 la cuota de un sabor de
# temporada se estima mezclando su pico con su valle y el contraste se dispara en falso.
def local_share(values, exposure, ref_idx):
    return values[ref_idx].sum() / max(exposure[ref_idx].sum(), 1.0)

candidates = []
for flavor in FLAVORS:
    in_catalog = alive[flavor].to_numpy()
    f_lines, f_units = shop_lines[flavor].to_numpy(), shop_units[flavor].to_numpy()
    f_club = club_units[flavor].to_numpy()
    x_lines = (shop_lines.sum(axis=1) - shop_lines[flavor]).to_numpy()
    x_units = (shop_units.sum(axis=1) - shop_units[flavor]).to_numpy()
    x_club = (club_units.sum(axis=1) - club_units[flavor]).to_numpy()

    for i, j in zero_runs(shop_lines[flavor] == 0, alive[flavor], MIN_RUN_DAYS):
        lo, hi = max(0, i - REF_DAYS), min(len(CALENDAR), j + 1 + REF_DAYS)
        ref = np.r_[np.arange(lo, i), np.arange(j + 1, hi)]
        ref = ref[in_catalog[ref]]
        if len(ref) < 20:
            continue
        win = np.arange(i, j + 1)
        share_lines = local_share(f_lines, x_lines, ref)
        share_units = local_share(f_units, x_units, ref)
        share_club = local_share(f_club, x_club, ref)
        expected_lines = share_lines * x_lines[win].sum()
        club_expected = share_club * x_club[win].sum()
        candidates.append({
            "flavor": flavor, "start": CALENDAR[i], "end": CALENDAR[j], "days": j - i + 1,
            "expected_lines": expected_lines,
            "expected_units": share_units * x_units[win].sum(),
            "share_units": share_units,
            "p_value": float(poisson.pmf(0, expected_lines)),
            "club_observed": float(f_club[win].sum()),
            "club_expected": float(club_expected),
            "club_ratio": float(f_club[win].sum() / club_expected) if club_expected > 0 else np.nan,
        })

candidates = pd.DataFrame(candidates).sort_values("p_value").reset_index(drop=True)
# Bonferroni sobre todas las rachas contrastadas: se buscan unas pocas ventanas entre decenas de
# candidatos, así que sin corregir la multiplicidad el nivel nominal no significa nada.
candidates["p_adjusted"] = (candidates.p_value * len(candidates)).clip(upper=1.0)
candidates["significant"] = candidates.p_adjusted < STOCKOUT_ALPHA
candidates["material"] = candidates.expected_lines >= MIN_EXPECTED_LINES
candidates["is_stockout"] = candidates.significant & candidates.material

print(f"{len(candidates)} rachas contrastadas · {int(candidates.significant.sum())} significativas · "
      f"{int(candidates.is_stockout.sum())} significativas Y materiales")
print()
print(candidates.head(10)[["flavor", "start", "end", "days", "expected_lines", "p_adjusted",
                           "club_observed", "club_expected", "club_ratio", "is_stockout"]]
      .assign(start=lambda d: d.start.dt.date, end=lambda d: d.end.dt.date)
      .round(3).to_string(index=False))

44 rachas contrastadas · 8 significativas · 6 significativas Y materiales

              flavor      start        end  days  expected_lines  p_adjusted  club_observed  club_expected  club_ratio  is_stockout
          intense_10 2026-04-24 2026-05-17    24         420.753       0.000          397.0        423.974       0.936         True
       decaf_vanilla 2026-06-22 2026-07-08    17         217.842       0.000          200.0        214.488       0.932         True
       decaf_vanilla 2025-03-09 2025-03-26    18         101.021       0.000          144.0        139.049       1.036         True
       decaf_vanilla 2025-06-29 2025-07-18    20          92.590       0.000          151.0        165.726       0.911         True
    seasonal_pumpkin 2026-02-25 2026-03-14    18          79.658       0.000           36.0         25.118       1.433         True
     origin_colombia 2024-07-06 2024-07-31    26          60.608       0.000          249.0        216.623       1.149         True
s

El contraste es directo: si la demanda diaria del sabor se comporta como un Poisson de tasa local
λ(d), la probabilidad de ver **cero líneas durante toda la ventana** es exp(−Σλ(d)). Con 44 rachas
contrastadas se corrige por Bonferroni.

Eso deja ocho ventanas significativas, y las dos últimas sobran: son rachas de cinco días sobre
sabores de temporada **en su valle**, donde ni siquiera una ventana de referencia corta consigue
limpiar del todo el contagio del pico. Se descartan con un criterio que además es el que usaría el
negocio: **materialidad**. Por debajo de 20 líneas de demanda censurada no hay un problema de
inventario que gestionar, haya o no significación estadística. Las seis que quedan censuran entre 61
y 421 líneas; las descartadas, 12 y 17.

La columna `club_ratio` es la confirmación independiente, y es la que distingue una rotura de stock
de un sabor que dejó de gustar: **durante las seis ventanas el club siguió sirviendo ese sabor con
normalidad** (ratio entre 0,91 y 1,43 sobre lo esperado) mientras la tienda marcaba cero. Si el
problema fuera de demanda, caerían los dos canales. Cae uno solo, y es exactamente el que
`docs/data_imperfections.md` dice que se censura.

In [8]:
stockouts = candidates[candidates.is_stockout].sort_values("start").reset_index(drop=True)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Tienda — líneas de pedido diarias (media móvil 7d)",
                                    "Club — unidades enviadas diarias (media móvil 7d)"))
focus = "decaf_vanilla"
fig.add_trace(go.Scatter(x=CALENDAR, y=shop_lines[focus].rolling(7).mean(), mode="lines",
                         line=dict(color=C_BLUE, width=1.8), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=CALENDAR, y=club_units[focus].rolling(7).mean(), mode="lines",
                         line=dict(color=C_AQUA, width=1.8), showlegend=False), row=2, col=1)
for row in stockouts[stockouts.flavor == focus].itertuples():
    for r in (1, 2):
        fig.add_vrect(x0=row.start, x1=row.end, fillcolor=C_RED, opacity=0.16,
                      line_width=0, row=r, col=1)
fig.update_layout(**{**PLOT_LAYOUT, "height": 460},
                  title=f"{FLAVOR_LABEL[focus]}: la rotura se ve en tienda y no en el club")
fig.update_yaxes(gridcolor=C_GRID)
fig.show()

print(f"{FLAVOR_LABEL[focus]} — tres ventanas detectadas:")
print(stockouts[stockouts.flavor == focus][["start", "end", "days", "expected_lines",
                                            "club_observed", "club_expected"]]
      .assign(start=lambda d: d.start.dt.date, end=lambda d: d.end.dt.date)
      .round(1).to_string(index=False))

Descaf. vainilla — tres ventanas detectadas:
     start        end  days  expected_lines  club_observed  club_expected
2025-03-09 2025-03-26    18           101.0          144.0          139.0
2025-06-29 2025-07-18    20            92.6          151.0          165.7
2026-06-22 2026-07-08    17           217.8          200.0          214.5


In [9]:
# Validación contra el ground truth del generador. El manifiesto NO se ha usado para detectar:
# no se carga en DuckDB y el procedimiento anterior sólo mira los marts. Se abre ahora, una vez
# cerrada la detección, para medir si el procedimiento acierta y si la imputación acierta el tamaño.
manifest = json.loads((PROJECT_ROOT / "data" / "raw" / "_imperfections_manifest.json")
                      .read_text(encoding="utf-8"))
truth = pd.DataFrame(manifest["stockouts"])
truth["start"] = pd.to_datetime(truth.start)

found = {(r.flavor, r.start) for r in stockouts.itertuples()}
real = {(r.flavor, r.start) for r in truth.itertuples()}
print(f"ventanas reales: {len(real)} · detectadas: {len(found)} · "
      f"aciertos: {len(found & real)} · falsos positivos: {len(found - real)} · "
      f"no detectadas: {len(real - found)}")
print()

check = (truth.merge(stockouts[["flavor", "start", "expected_lines"]], on=["flavor", "start"],
                     how="left")
              .rename(columns={"censored_order_lines": "lineas_reales",
                               "expected_lines": "lineas_estimadas"}))
check["error_pct"] = (check.lineas_estimadas / check.lineas_reales - 1) * 100
print(check[["flavor", "start", "end", "lineas_reales", "lineas_estimadas", "error_pct"]]
      .assign(start=lambda d: d.start.dt.date).round(1).to_string(index=False))
print()
print(f"total real {check.lineas_reales.sum():.0f} líneas · "
      f"estimado {check.lineas_estimadas.sum():.1f} · "
      f"error {check.lineas_estimadas.sum() / check.lineas_reales.sum() - 1:+.2%}")

ventanas reales: 6 · detectadas: 6 · aciertos: 6 · falsos positivos: 0 · no detectadas: 0

          flavor      start        end  lineas_reales  lineas_estimadas  error_pct
   decaf_vanilla 2026-06-22 2026-07-08            199             217.8        9.5
   decaf_vanilla 2025-03-09 2025-03-26            104             101.0       -2.9
 origin_colombia 2024-07-06 2024-07-31             46              60.6       31.8
   decaf_vanilla 2025-06-29 2025-07-18             96              92.6       -3.6
seasonal_pumpkin 2026-02-25 2026-03-14             76              79.7        4.8
      intense_10 2026-04-24 2026-05-17            445             420.8       -5.4

total real 966 líneas · estimado 972.5 · error +0.67%


**Seis de seis, sin falsos positivos**, y el tamaño estimado de la censura se desvía un **+0,7%** del
real agregado (972 líneas frente a 966). Ventana a ventana el error es mayor —Colombia se sobreestima
un 32% sobre una base pequeña— pero para reconstruir la serie lo que importa es el agregado, y ahí la
estimación es casi exacta.

Que el procedimiento acierte no es la parte interesante: es la comprobación de que el criterio con el
que se corrige la serie no es arbitrario. La parte interesante es **qué hace falta para acertar**:
una tasa esperada en vez de una racha, una ventana de referencia corta para no contagiar la
estacionalidad, una corrección por multiplicidad, un umbral de materialidad y un canal de control que
no esté afectado por la imperfección.

In [10]:
# Imputación: la demanda censurada se reconstruye con la cuota local del sabor sobre el resto del
# catálogo. No es "rellenar con la media": el multiplicador es lo que vendieron los demás esos
# mismos días, así que hereda el día de la semana, la tendencia y cualquier campaña.
shop_units_adj = shop_units.copy()
for row in stockouts.itertuples():
    window = (CALENDAR >= row.start) & (CALENDAR <= row.end)
    exposure = (shop_units.sum(axis=1) - shop_units[row.flavor]).to_numpy()
    shop_units_adj.loc[window, row.flavor] = row.share_units * exposure[window]

avg_price = shop_eur.sum() / shop_units.sum()
imputed_units = (shop_units_adj - shop_units).sum()
lost = pd.DataFrame({"unidades_censuradas": imputed_units,
                     "precio_medio_eur": avg_price,
                     "euros_censurados": imputed_units * avg_price})
lost = lost[lost.unidades_censuradas > 0].sort_values("euros_censurados", ascending=False)
print(lost.round(2).to_string())
print()
print(f"demanda censurada total: {imputed_units.sum():,.0f} unidades · "
      f"{(imputed_units * avg_price).sum():,.0f} € "
      f"({(imputed_units * avg_price).sum() / shop_eur.to_numpy().sum():.2%} "
      f"de la facturación de cápsulas en tienda)")

demand_daily = shop_units_adj + club_units
monthly = demand_daily.resample("MS").sum()
monthly_raw = demand_daily_raw.resample("MS").sum()
impact = pd.DataFrame({"censurada": monthly_raw.sum(axis=1), "corregida": monthly.sum(axis=1)})
impact["dif_pct"] = (impact.corregida / impact.censurada - 1) * 100
print()
print("Meses afectados:")
print(impact[impact.dif_pct > 0.05].round(2).to_string())

                  unidades_censuradas  precio_medio_eur  euros_censurados
flavor                                                                   
decaf_vanilla                 1537.24              4.91           7551.48
intense_10                    1552.25              4.46           6929.34
seasonal_pumpkin               277.83              5.90           1637.88
origin_colombia                226.22              4.42            999.95

demanda censurada total: 3,594 unidades · 17,119 € (1.35% de la facturación de cápsulas en tienda)

Meses afectados:
            censurada  corregida  dif_pct
2024-07-01     3587.0    3813.22     6.31
2025-03-01     9941.0   10309.15     3.70
2025-06-01    10058.0   10085.31     0.27
2025-07-01     9270.0    9581.07     3.36
2026-02-01    18670.0   18728.55     0.31
2026-03-01    22287.0   22506.27     0.98
2026-04-01    20623.0   21084.92     2.24
2026-05-01    21207.0   22297.34     5.14
2026-06-01    20461.0   20949.27     2.39
2026-07-01    1851

La censura vale **3.594 unidades y 17.119 €**, un 1,35% de la facturación de cápsulas en tienda. Como
cifra agregada es pequeña; como problema de serie temporal, no, por dónde cae:

**Tres de las seis ventanas caen en los últimos siete meses del histórico**, y una de ellas —Intenso
10, del 24 de abril al 17 de mayo de 2026— es la mayor de todas. Mayo de 2026 aparece un 5,1% por
debajo de su demanda real y junio un 2,4%. Esos son justo los meses que fijan el origen del forecast.

Es el mismo patrón que apareció en la página de ingresos con el vertido de eventos sobre el último
día: **una imperfección pequeña en el agregado puede ser grande en el sitio exacto donde arranca la
previsión**. La sección 6 mide cuánto.

## 4. La serie de demanda: un agregado y doce series

Con la censura corregida y los SKUs unidos por sabor, ya hay serie que analizar. Y a diferencia de
las dos páginas anteriores, aquí no hay una serie: hay trece —el total y sus doce componentes— y la
pregunta de diseño es si merece la pena mirarlas por separado.

In [11]:
total_demand = ts.build_series(
    demand_daily.sum(axis=1).rename_axis("d").reset_index(name="units"),
    "d", "units", freq="MS")
total_demand.name = "units"

share_total = (monthly.sum() / monthly.sum().sum() * 100).sort_values(ascending=False)
share_recent = (monthly.iloc[-12:].sum() / monthly.iloc[-12:].sum().sum() * 100)
mix = pd.DataFrame({"cuota_historica_pct": share_total,
                    "cuota_12m_pct": share_recent.reindex(share_total.index),
                    "unidades": monthly.sum().reindex(share_total.index)})
print(mix.round(2).to_string())
print()
print(f"histórico: {len(total_demand)} meses · {total_demand.sum():,.0f} unidades · "
      f"último mes {total_demand.iloc[-1]:,.0f}")

fig = go.Figure()
for flavor in share_total.index[::-1]:
    fig.add_trace(go.Scatter(x=monthly.index, y=monthly[flavor], name=FLAVOR_LABEL[flavor],
                             mode="lines", stackgroup="one",
                             line=dict(color=FLAVOR_COLOR[flavor], width=0.5)))
fig.update_layout(**{**PLOT_LAYOUT, "height": 480, "legend": dict(orientation="v", x=1.01, y=1)},
                  title="Demanda mensual de cápsulas por sabor (club + tienda, unidades)",
                  yaxis_title="unidades / mes")
fig.show()

                      cuota_historica_pct  cuota_12m_pct  unidades
flavor                                                            
classic_espresso                    15.16          14.86  55950.00
origin_colombia                     13.11          13.01  48368.22
intense_10                          10.71          10.58  39527.25
intense_8                            9.67           9.72  35698.00
origin_brazil                        9.33           9.06  34443.00
decaf_classic                        8.60           8.60  31749.00
decaf_vanilla                        8.41           8.36  31031.24
classic_lungo                        6.45           6.31  23785.00
seasonal_pumpkin                     6.18           6.85  22802.83
origin_ethiopia                      5.01           4.82  18486.00
intense_12                           4.51           4.43  16657.00
seasonal_gingerbread                 2.86           3.41  10546.00

histórico: 36 meses · 369,044 unidades · último mes 18,277


In [12]:
decomposition = ts.stl_decompose(total_demand, period=SEASON_LENGTH, robust=True)
print(f"TOTAL  F_tendencia={decomposition.trend_strength:.3f}  "
      f"F_estacional={decomposition.seasonal_strength[SEASON_LENGTH]:.3f}")

strength_rows = []
for flavor in FLAVORS:
    s = monthly[flavor][monthly[flavor] > 0].asfreq("MS")
    if len(s) < 2 * SEASON_LENGTH:
        strength_rows.append({"sabor": flavor, "meses": len(s), "F_tendencia": np.nan,
                              "F_estacional": np.nan, "nota": "histórico < 2 ciclos"})
        continue
    d = ts.stl_decompose(s, period=SEASON_LENGTH, robust=True)
    strength_rows.append({"sabor": flavor, "meses": len(s), "F_tendencia": d.trend_strength,
                          "F_estacional": d.seasonal_strength[SEASON_LENGTH], "nota": ""})
strength = pd.DataFrame(strength_rows).sort_values("F_estacional", ascending=False)
print()
print(strength.round(3).to_string(index=False))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=("Tendencia", "Estacionalidad anual", "Residuo"))
for row, comp in enumerate([decomposition.trend,
                            decomposition.seasonal[SEASON_LENGTH],
                            decomposition.resid], start=1):
    fig.add_trace(go.Scatter(x=comp.index, y=comp.to_numpy(), mode="lines",
                             line=dict(color=C_BLUE, width=2), showlegend=False), row=row, col=1)
fig.add_hline(y=0, line=dict(color=C_MUTED, width=1), row=3, col=1)
fig.update_layout(**{**PLOT_LAYOUT, "height": 560},
                  title="STL de la demanda total de cápsulas")
fig.show()

TOTAL  F_tendencia=0.994  F_estacional=0.911



               sabor  meses  F_tendencia  F_estacional                 nota
    seasonal_pumpkin     30        0.855         0.958                     
       classic_lungo     36        0.991         0.867                     
          intense_12     36        0.988         0.855                     
       origin_brazil     36        0.991         0.852                     
    classic_espresso     36        0.988         0.852                     
       decaf_classic     36        0.987         0.839                     
     origin_ethiopia     36        0.993         0.817                     
     origin_colombia     36        0.990         0.804                     
           intense_8     36        0.982         0.773                     
       decaf_vanilla     36        0.983         0.771                     
          intense_10     36        0.974         0.725                     
seasonal_gingerbread     21          NaN           NaN histórico < 2 ciclos


In [13]:
# Perfil estacional por sabor: índice demanda / tendencia, en mediana y sólo sobre el tramo maduro.
# Misma precaución que en la página de ingresos: en el año de arranque la tendencia es tan pequeña
# que el cociente se dispara.
PROFILE_MONTHS = 24
profiles = {}
for flavor in FLAVORS:
    s = monthly[flavor][monthly[flavor] > 0].asfreq("MS")
    trend = s.rolling(SEASON_LENGTH, center=True, min_periods=SEASON_LENGTH // 2).mean()
    index = (s / trend).iloc[-PROFILE_MONTHS:]
    profiles[flavor] = index.groupby(index.index.month).median().reindex(range(1, 13))
profile_frame = pd.DataFrame(profiles)
print("Índice estacional (1,00 = mes en línea con la tendencia):")
print(profile_frame.round(2).to_string())

fig = go.Figure(go.Heatmap(
    z=profile_frame.T.to_numpy(), x=MONTHS,
    y=[FLAVOR_LABEL[f] for f in profile_frame.columns],
    colorscale=[[0, "#2a78d6"], [0.5, "#f4f4f2"], [1, "#eb6834"]], zmid=1.0,
    colorbar=dict(title="índice"), hovertemplate="%{y} · %{x}: %{z:.2f}<extra></extra>"))
fig.update_layout(**{**PLOT_LAYOUT, "height": 420, "hovermode": "closest"},
                  title="Perfil estacional por sabor (mediana de demanda / tendencia)")
fig.show()

base = [f for f in FLAVORS if not f.startswith("seasonal_")]
print()
print("Sabores de base: mes techo y mes suelo del perfil medio")
base_profile = profile_frame[base].mean(axis=1)
print(f"  techo {MONTHS[int(base_profile.idxmax()) - 1]} ({base_profile.max():.2f}) · "
      f"suelo {MONTHS[int(base_profile.idxmin()) - 1]} ({base_profile.min():.2f}) · "
      f"amplitud {(base_profile.max() / base_profile.min() - 1) * 100:.0f}%")
for flavor in ("seasonal_pumpkin", "seasonal_gingerbread"):
    p = profile_frame[flavor]
    print(f"  {FLAVOR_LABEL[flavor]:20s} techo {MONTHS[int(p.idxmax()) - 1]} ({p.max():.2f}) · "
          f"suelo {MONTHS[int(p.idxmin()) - 1]} ({p.min():.2f}) · "
          f"amplitud {(p.max() / p.min() - 1) * 100:.0f}%")

Índice estacional (1,00 = mes en línea con la tendencia):
    classic_espresso  classic_lungo  decaf_classic  decaf_vanilla  intense_10  intense_12  intense_8  origin_brazil  origin_colombia  origin_ethiopia  seasonal_gingerbread  seasonal_pumpkin
1               1.19           1.05           1.23           1.11        1.29        1.17       1.16           1.16             1.09             1.11                  1.94              0.29
2               1.15           1.10           1.11           1.14        1.10        1.08       1.15           1.11             1.04             1.15                  0.36              0.33
3               1.27           1.34           1.19           1.24        1.26        1.19       1.20           1.23             1.22             1.18                  0.40              0.44
4               1.16           1.04           1.20           1.10        1.22        1.20       1.16           1.09             1.10             1.10                  0.37           


Sabores de base: mes techo y mes suelo del perfil medio
  techo Mar (1.23) · suelo Ago (0.84) · amplitud 47%
  Temporada calabaza   techo Oct (3.47) · suelo Ene (0.29) · amplitud 1104%
  Temporada jengibre   techo Dic (3.91) · suelo Sep (0.14) · amplitud 2618%


**Hay dos regímenes de demanda, y sólo uno se ve en el agregado.**

Los diez sabores de base se comportan como un único producto repartido en diez etiquetas: todos hacen
techo en marzo y suelo en agosto, con una amplitud pico-valle del 47%. Es el ciclo del café —se
consume menos en verano— y es el que hereda el agregado.

Los dos de temporada son otra cosa. Calabaza multiplica por 3,5 su tendencia en octubre y cae a 0,29
en enero —**una amplitud de doce a uno**— y Jengibre, que concentra su año entre noviembre y enero,
llega a veintisiete a uno entre diciembre y septiembre. Juntos son el 9% del volumen, así que en la serie total son invisibles; pero son los que deciden el
calendario de compras, y son también los dos SKUs cuya previsión tiene que salir bien en dos meses
concretos del año o no sirve de nada.

Ésta es la razón de ser de una página a nivel de SKU: **el agregado no es una versión resumida de las
partes, es una mezcla que borra justo la parte que hay que planificar.**

In [14]:
# Estacionalidad semanal: aquí el grano diario sí existe en las dos mitades, y no se comportan igual.
shop_daily_total = ts.build_series(
    shop_units_adj.sum(axis=1).rename_axis("d").reset_index(name="units"), "d", "units",
    freq="D", start=CALENDAR[0], end=CALENDAR[-1])
club_daily_total = ts.build_series(
    club_units.sum(axis=1).rename_axis("d").reset_index(name="units"), "d", "units",
    freq="D", start=CALENDAR[0], end=CALENDAR[-1])

weekly = {}
for name, series in (("tienda", shop_daily_total), ("club", club_daily_total)):
    d = ts.mstl_decompose(series, periods=ts.infer_seasonal_periods(series))
    comp = d.seasonal[7]
    profile = comp.groupby(comp.index.dayofweek).mean() / series.mean() * 100
    weekly[name] = {"decomposition": d, "profile": profile}
    print(f"{name:7s} F_semanal={d.seasonal_strength[7]:.3f} "
          f"F_anual={d.seasonal_strength.get(365, float('nan')):.3f} "
          f"F_tendencia={d.trend_strength:.3f} media={series.mean():.1f} u/día")
    print("        " + "  ".join(f"{DAYS[i]} {profile.iloc[i]:+.0f}%" for i in range(7)))

fig = go.Figure()
for name, color in (("tienda", C_BLUE), ("club", C_AQUA)):
    fig.add_trace(go.Bar(x=DAYS, y=weekly[name]["profile"].to_numpy(),
                         name=name.capitalize(), marker_color=color))
fig.add_hline(y=0, line=dict(color=C_MUTED, width=1))
fig.update_layout(**{**PLOT_LAYOUT, "height": 380, "barmode": "group"},
                  title="Componente semanal de la demanda, en % sobre la media del canal",
                  yaxis_title="% sobre la media")
fig.show()

tienda  F_semanal=0.627 F_anual=0.767 F_tendencia=0.961 media=233.0 u/día
        Lun +14%  Mar +10%  Mié +8%  Jue +8%  Vie +5%  Sáb -15%  Dom -29%


club    F_semanal=0.220 F_anual=0.684 F_tendencia=0.968 media=103.8 u/día
        Lun -4%  Mar +1%  Mié +4%  Jue +1%  Vie -1%  Sáb +1%  Dom -2%


**El club no tiene semana y la tienda sí.** La fuerza de la estacionalidad semanal es 0,22 en el club
frente a 0,63 en tienda, y el perfil lo explica: los envíos del club se reparten de forma casi plana
(±4%) porque los programa el almacén, mientras que la tienda cae un 29% el domingo respecto a la
media y hace máximo el lunes.

Tiene una consecuencia práctica inmediata: **el 31% de la demanda que va por el club es la parte
amortiguable**. Si el cuello de botella semanal está en el pico de lunes-martes, el margen de
maniobra está en mover envíos del club, no en la tienda, que es la que impone el perfil.

Es también el complemento exacto de lo que vio la página de ingresos, que sólo pudo medir la semana
sobre la parte de tienda porque la suscripción se factura por ciclos mensuales. En unidades físicas
el club sí tiene grano diario, y resulta que es plano.

## 5. Estacionariedad

In [15]:
MAXLAG = int(np.ceil(4 * (len(total_demand) / 100) ** (2 / 9)))
stat_kwargs = dict(season_length=SEASON_LENGTH, alpha=ALPHA, regression="ct", maxlag=MAXLAG)
report_level = ts.stationarity_report(total_demand, **stat_kwargs)
report_log = ts.stationarity_report(np.log(total_demand), **stat_kwargs)
print(f"maxlag = {MAXLAG} (acotado: con {len(total_demand)} puntos la regla por defecto pediría "
      f"{int(np.ceil(12 * (len(total_demand) / 100) ** 0.25))})")
print()
print("Serie en nivel :", report_level.summary())
print("Serie en log   :", report_log.summary())
print()
print("Por sabor (log), con la misma configuración:")
flavor_stationarity = {}
for flavor in FLAVORS:
    s = monthly[flavor][monthly[flavor] > 0].asfreq("MS")
    if len(s) < 24:
        print(f"  {FLAVOR_LABEL[flavor]:20s} histórico corto ({len(s)} meses), no se contrasta")
        continue
    r = ts.stationarity_report(np.log(s.clip(lower=1)), **stat_kwargs)
    flavor_stationarity[flavor] = r
    print(f"  {FLAVOR_LABEL[flavor]:20s} d={r.n_diffs} D={r.n_seasonal_diffs}  "
          f"ADF p={r.adf.p_value:.3f}")

maxlag = 4 (acotado: con 36 puntos la regla por defecto pediría 10)

Serie en nivel : discrepancia: ADF no rechaza la raíz unitaria pero KPSS no rechaza la estacionariedad | ADF p=0.2343 | KPSS p=0.1000 (recortado) | d=1, D=1 (m=12)
Serie en log   : no estacionaria (ADF y KPSS coinciden) | ADF p=0.5209 | KPSS p=0.0120 | d=0, D=1 (m=12)

Por sabor (log), con la misma configuración:
  Espresso clásico     d=0 D=1  ADF p=0.023
  Lungo clásico        d=0 D=0  ADF p=0.356
  Descafeinado         d=1 D=1  ADF p=0.403
  Descaf. vainilla     d=0 D=1  ADF p=0.007
  Intenso 10           d=1 D=1  ADF p=0.344
  Intenso 12           d=0 D=1  ADF p=0.094
  Intenso 8            d=1 D=1  ADF p=0.549
  Origen Brasil        d=0 D=0  ADF p=0.003
  Origen Colombia      d=0 D=1  ADF p=0.345
  Origen Etiopía       d=0 D=1  ADF p=0.051
  Temporada jengibre   histórico corto (21 meses), no se contrasta


  Temporada calabaza   d=0 D=1  ADF p=0.100


El veredicto es el mismo que en las dos páginas anteriores y por la misma razón: 36 puntos, tendencia
fuerte y dos contrastes con poca potencia. Se modela en **logaritmos** —la demanda crece de forma
multiplicativa— con una diferencia regular y una estacional. Los sabores individuales coinciden en el
diagnóstico, lo que ya adelanta que un mismo orden SARIMA sirve para los doce.

## 6. Modelos y backtesting

En las dos páginas anteriores la pregunta era si desagregar mejora el agregado. Con doce series en
vez de una, la pregunta tiene dos direcciones:

- **Bottom-up**: un modelo por sabor, y el total es su suma.
- **Top-down**: un modelo sobre el total, repartido entre sabores por su cuota.

Los dos son defendibles a priori. Un sabor es un 8% del volumen y por tanto tiene mucho más ruido
relativo que el total, lo que favorece al top-down; pero cada sabor tiene su propio calendario, y
eso favorece al bottom-up.

| Modelo | Qué hace |
|---|---|
| `naive_estacional` | Repite el mismo mes del año anterior. El suelo de referencia. |
| `sarima_airline` | Un SARIMA(0,1,1)(0,1,1)₁₂ sobre el log de **esa** serie. |
| `top_down_cuota` | Pronostica el **total** con SARIMA y lo reparte con el perfil de cuota por mes natural de cada sabor. |

El reparto del `top_down` no es una cuota fija: es la **mediana de la cuota de ese sabor en ese mes
natural** sobre los últimos 24 meses del train. Sin el mes natural, Calabaza recibiría su 6% medio
también en octubre, cuando le toca el 24%.

In [16]:
def sarima_forecast(train, horizon, order=SARIMA_ORDER, seasonal=SARIMA_SEASONAL,
                    level=INTERVAL_LEVEL):
    """SARIMA sobre log de la demanda. Devuelve punto e intervalo en unidades."""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    values = pd.Series(np.log(np.asarray(train, dtype=float).clip(min=1.0)), index=train.index)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = SARIMAX(values, order=order, seasonal_order=seasonal,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        forecast = fitted.get_forecast(horizon)
        conf = forecast.conf_int(alpha=1 - level)
    return pd.DataFrame({"yhat": np.exp(forecast.predicted_mean.to_numpy()),
                         "yhat_lower": np.exp(conf.iloc[:, 0].to_numpy()),
                         "yhat_upper": np.exp(conf.iloc[:, 1].to_numpy())})
sarima_forecast.__name__ = "sarima_airline"

SHARE_MONTHS = 24
def share_profile(matrix, cutoff):
    """Cuota de cada columna por mes natural, estimada SÓLO con datos hasta `cutoff`."""
    history = matrix.loc[:cutoff].iloc[-SHARE_MONTHS:]
    shares = history.div(history.sum(axis=1), axis=0)
    profile = shares.groupby(shares.index.month).median()
    return profile.div(profile.sum(axis=1), axis=0)   # renormalizado: las partes suman el total

def future_months_of(cutoff, horizon):
    return pd.date_range(cutoff + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")

def make_top_down(flavor):
    def _forecast(train, horizon):
        cutoff = train.index[-1]
        total_hat = sarima_forecast(monthly.loc[:cutoff].sum(axis=1).asfreq("MS"), horizon)["yhat"]
        months = future_months_of(cutoff, horizon).month
        return total_hat.to_numpy() * share_profile(monthly, cutoff)[flavor].reindex(months).to_numpy()
    _forecast.__name__ = "top_down_cuota"
    return _forecast

MIN_MONTHS_SARIMA = 2 * SEASON_LENGTH + 2   # dos ciclos completos + margen para diferenciar

flavor_backtests, per_flavor_rows = {}, []
for flavor in FLAVORS:
    series = monthly[flavor][monthly[flavor] > 0].asfreq("MS")
    folds = BACKTEST_FOLDS if len(series) >= MIN_MONTHS_SARIMA else 3
    row = {"sabor": flavor, "meses": len(series)}
    flavor_backtests[flavor] = {}
    for label, fn in (("naive_estacional", ts.make_seasonal_naive(SEASON_LENGTH)),
                      ("sarima_airline", sarima_forecast),
                      ("top_down_cuota", make_top_down(flavor))):
        if label == "sarima_airline" and len(series) < MIN_MONTHS_SARIMA:
            row[f"mase_{label}"] = np.nan
            row[f"mape_{label}"] = np.nan
            continue
        result = ts.walk_forward_backtest(series, fn, horizon=FORECAST_HORIZON, n_folds=folds,
                                          step=1, season_length=SEASON_LENGTH,
                                          model_name=label, on_error="skip")
        flavor_backtests[flavor][label] = result
        row[f"mase_{label}"] = result.metrics["mase"]
        row[f"mape_{label}"] = result.metrics["mape"]
    per_flavor_rows.append(row)

per_flavor = pd.DataFrame(per_flavor_rows)
print(per_flavor.round(3).to_string(index=False))
print()
print("MASE medio  naive:", round(per_flavor.mase_naive_estacional.mean(), 3),
      "· sarima_airline:", round(per_flavor.mase_sarima_airline.mean(), 3),
      "· top_down_cuota:", round(per_flavor.mase_top_down_cuota.mean(), 3))

               sabor  meses  mase_naive_estacional  mape_naive_estacional  mase_sarima_airline  mape_sarima_airline  mase_top_down_cuota  mape_top_down_cuota
    classic_espresso     36                  1.720                 52.210                0.495               14.831                0.543               16.446
       classic_lungo     36                  1.528                 50.699                0.683               22.972                0.447               15.109
       decaf_classic     36                  1.591                 52.071                0.594               19.203                0.398               13.017
       decaf_vanilla     36                  1.792                 54.034                0.522               15.677                0.580               17.502
          intense_10     36                  1.645                 52.025                0.305                9.632                0.453               14.489
          intense_12     36                  1.624  

**En los sabores, empate técnico entre las dos direcciones**: 0,478 de MASE medio el bottom-up puro
contra 0,450 el top-down, repartiéndose los sabores casi mitad y mitad.

Pero hay una asimetría que la media esconde. **Jengibre no tiene histórico para un SARIMA
estacional** —nació en diciembre de 2024, tiene 21 meses y hacen falta dos ciclos completos para
diferenciar estacionalmente— y el top-down sí puede pronosticarlo, porque le presta al total la forma
del ciclo y sólo tiene que estimar su cuota. En un catálogo vivo ése es justo el caso que más falta
hace prever: los SKUs nuevos son los que menos histórico tienen.

Y el naive estacional, que en las páginas anteriores era un suelo razonable, aquí es inservible:
MASE 1,41 de media, **peor que repetir el último valor**. En un negocio que casi dobla cada año,
repetir el mismo mes del año pasado es un error garantizado.

In [17]:
def bottom_up_forecast(train, horizon):
    """Suma de las previsiones por sabor. Sólo usa la serie total para fijar el corte."""
    cutoff = train.index[-1]
    total = np.zeros(horizon)
    for flavor in FLAVORS:
        s = monthly[flavor].loc[:cutoff]
        s = s[s > 0].asfreq("MS")
        if len(s) >= MIN_MONTHS_SARIMA:
            total += sarima_forecast(s, horizon)["yhat"].to_numpy()
        else:
            total += make_top_down(flavor)(s, horizon)
    return total
bottom_up_forecast.__name__ = "bottom_up_sabores"

total_models = {"naive_estacional": ts.make_seasonal_naive(SEASON_LENGTH),
                "sarima_airline": sarima_forecast,
                "bottom_up_sabores": bottom_up_forecast}
total_backtests = {
    name: ts.walk_forward_backtest(total_demand, fn, horizon=FORECAST_HORIZON,
                                   n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                   model_name=name, on_error="skip")
    for name, fn in total_models.items()}
print("Serie total, horizonte 6 meses, 5 pliegues:")
print(ts.compare_backtests(total_backtests)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

Serie total, horizonte 6 meses, 5 pliegues:
            model       mae   mape  mase       bias
bottom_up_sabores  1940.744  9.427 0.270   1465.707
   sarima_airline  2512.516 12.086 0.351    462.493
 naive_estacional 10844.045 52.497 1.493 -10844.045


**Y aquí el agregado dice lo contrario que en la página de ingresos.** Allí la suma de los canales no
llegaba a batir al SARIMA directo sobre el total; aquí la suma de los doce sabores lo bate con
holgura: MASE 0,270 frente a 0,351, y MAPE 9,4% frente a 12,1%.

No es una contradicción, es una diferencia real entre los dos problemas. Los cuatro canales de
ingreso comparten aproximadamente la misma forma estacional, así que agregar no destruía
información. Los doce sabores **no**: el total mezcla dos calendarios incompatibles —el ciclo del
café de los diez de base y el ciclo de temporada de Calabaza y Jengibre—, y un único SARIMA sobre esa
mezcla tiene que ajustar una estacionalidad que no existe en ninguna de sus partes.

Lo que sugiere que ninguna de las dos direcciones puras es la correcta.

### Middle-out: agrupar por calendario, no por tamaño

La sección 4 encontró dos regímenes de demanda. La consecuencia para el modelo es directa:

- **Los diez sabores de base comparten calendario.** Agregarlos en un bloque no pierde forma
  estacional y sí cancela mucho ruido, así que ese bloque se pronostica junto y se reparte top-down
  con la cuota dentro del bloque —que es estable, porque no tiene que absorber los picos de
  temporada.
- **Los dos de temporada tienen calendario propio.** Se pronostican por separado: Calabaza con su
  SARIMA, que ya tiene 30 meses, y Jengibre con el reparto del total, que es lo único que le da forma
  de ciclo con 21.

Es un **middle-out**: se elige el nivel de agregación por homogeneidad estacional en vez de por
posición en la jerarquía.

In [18]:
BASE_FLAVORS = [f for f in FLAVORS if not f.startswith("seasonal_")]
SEASONAL_FLAVORS = [f for f in FLAVORS if f.startswith("seasonal_")]

_middle_out_cache = {}
def middle_out_parts(cutoff, horizon):
    """Previsión de los doce sabores a la vez. Se cachea porque el backtesting por sabor la
    pediría doce veces con el mismo corte y las doce devolverían lo mismo."""
    key = (cutoff, horizon)
    if key in _middle_out_cache:
        return _middle_out_cache[key]
    months = future_months_of(cutoff, horizon).month
    parts = {}
    block = monthly[BASE_FLAVORS].loc[:cutoff].sum(axis=1).asfreq("MS")
    block_hat = sarima_forecast(block, horizon)["yhat"].to_numpy()
    block_profile = share_profile(monthly[BASE_FLAVORS], cutoff)
    for flavor in BASE_FLAVORS:
        parts[flavor] = block_hat * block_profile[flavor].reindex(months).to_numpy()
    for flavor in SEASONAL_FLAVORS:
        s = monthly[flavor].loc[:cutoff]
        s = s[s > 0].asfreq("MS")
        parts[flavor] = (sarima_forecast(s, horizon)["yhat"].to_numpy()
                         if len(s) >= MIN_MONTHS_SARIMA
                         else make_top_down(flavor)(s, horizon))
    _middle_out_cache[key] = parts
    return parts

def make_middle_out(flavor):
    def _forecast(train, horizon):
        return middle_out_parts(train.index[-1], horizon)[flavor]
    _forecast.__name__ = "middle_out_grupos"
    return _forecast

def middle_out_total(train, horizon):
    return sum(middle_out_parts(train.index[-1], horizon).values())
middle_out_total.__name__ = "middle_out_grupos"

for flavor in FLAVORS:
    series = monthly[flavor][monthly[flavor] > 0].asfreq("MS")
    folds = BACKTEST_FOLDS if len(series) >= MIN_MONTHS_SARIMA else 3
    result = ts.walk_forward_backtest(series, make_middle_out(flavor), horizon=FORECAST_HORIZON,
                                      n_folds=folds, step=1, season_length=SEASON_LENGTH,
                                      model_name="middle_out_grupos", on_error="skip")
    flavor_backtests[flavor]["middle_out_grupos"] = result
    per_flavor.loc[per_flavor.sabor == flavor, "mase_middle_out_grupos"] = result.metrics["mase"]
    per_flavor.loc[per_flavor.sabor == flavor, "mape_middle_out_grupos"] = result.metrics["mape"]

MASE_COLS = ["mase_naive_estacional", "mase_sarima_airline", "mase_top_down_cuota",
             "mase_middle_out_grupos"]
per_flavor["gana"] = (per_flavor[MASE_COLS].idxmin(axis=1)
                      .str.replace("mase_", "", regex=False))
print(per_flavor[["sabor", "meses", *MASE_COLS, "gana"]].round(3).to_string(index=False))
print()
print("MASE medio por sabor:")
print(per_flavor[MASE_COLS].mean().round(3).to_string())

total_models["middle_out_grupos"] = middle_out_total
total_backtests["middle_out_grupos"] = ts.walk_forward_backtest(
    total_demand, middle_out_total, horizon=FORECAST_HORIZON, n_folds=BACKTEST_FOLDS, step=1,
    season_length=SEASON_LENGTH, model_name="middle_out_grupos", on_error="skip")
leaderboard = ts.compare_backtests(total_backtests)[
    ["model", "mae", "mape", "mase", "bias", "n_folds"]].round(3)
print()
print("Serie total, con el middle-out incluido:")
print(leaderboard.to_string(index=False))
print()
for name, result in total_backtests.items():
    print(f"{name:18s} MAPE por horizonte:",
          [round(r["mape"], 1) for r in result.metrics_by_horizon.to_dict("records")])

               sabor  meses  mase_naive_estacional  mase_sarima_airline  mase_top_down_cuota  mase_middle_out_grupos              gana
    classic_espresso     36                  1.720                0.495                0.543                   0.437 middle_out_grupos
       classic_lungo     36                  1.528                0.683                0.447                   0.324 middle_out_grupos
       decaf_classic     36                  1.591                0.594                0.398                   0.290 middle_out_grupos
       decaf_vanilla     36                  1.792                0.522                0.580                   0.494 middle_out_grupos
          intense_10     36                  1.645                0.305                0.453                   0.337    sarima_airline
          intense_12     36                  1.624                0.606                0.565                   0.527 middle_out_grupos
           intense_8     36                  1.822     

**El middle-out gana en los sabores, que es lo que esta página tiene que acertar**: MASE medio 0,384
frente a 0,450 del top-down y 0,478 del bottom-up —un 15% y un 20% menos de error— y es el mejor de
los cuatro modelos en **siete de los doce sabores**. En los otros cinco pierde por poco y por motivos
identificables: Intenso 10 y Brasil son los dos sabores de base más regulares y un SARIMA propio les
saca partido; Colombia queda a cinco milésimas del top-down; a Calabaza el SARIMA suelto le saca un
7%, porque en los pliegues más cortos el middle-out todavía tiene que repartirla desde el total en
vez de darle modelo propio; y Jengibre queda a la par del naive estacional (0,128 frente a 0,124),
que es el único sabor del catálogo donde ningún modelo gana con claridad.

En el total queda en 0,295, entre el bottom-up (0,270) y el SARIMA directo (0,351). Ese segundo
puesto en el agregado es un precio conocido y pequeño: el bottom-up gana el total porque deja a cada
sabor su propio modelo y los errores se le compensan al sumar, pero paga esa ventaja con un error por
sabor un 24% mayor. Para decidir cuántas cápsulas de cada sabor comprar, el error que importa es el
del sabor.

Además el middle-out es **coherente por construcción** —las doce previsiones suman exactamente la del
total, porque los dos repartos por cuota están renormalizados— y **cubre los doce SKUs**, incluido el
que no tiene histórico para un modelo estacional propio.

La lección general: **el nivel al que conviene modelar no lo fija la jerarquía del catálogo sino la
homogeneidad de la estacionalidad.** Agrupar diez sabores que comparten calendario cancela ruido sin
perder forma; meter en ese mismo saco a Calabaza y Jengibre destruye justo la información que
distingue a esos dos SKUs.

In [19]:
# ¿Cuánto vale haber tratado la censura? Mismo objetivo de evaluación (la serie corregida, que es la
# mejor estimación de la demanda real) y dos entrenamientos distintos: con la serie tal cual está en
# el mart, y con la serie corregida.
def trained_on_censored(raw_series):
    def _forecast(train, horizon):
        return sarima_forecast(raw_series.reindex(train.index), horizon)
    _forecast.__name__ = "sarima_sobre_serie_censurada"
    return _forecast

censoring_rows = []
for name in sorted(stockouts.flavor.unique()) + ["TOTAL"]:
    clean = total_demand if name == "TOTAL" else monthly[name][monthly[name] > 0].asfreq("MS")
    raw = (monthly_raw.sum(axis=1) if name == "TOTAL" else monthly_raw[name]).reindex(clean.index)
    r_clean = ts.walk_forward_backtest(clean, sarima_forecast, horizon=FORECAST_HORIZON,
                                       n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                       model_name="corregida", on_error="skip")
    r_raw = ts.walk_forward_backtest(clean, trained_on_censored(raw), horizon=FORECAST_HORIZON,
                                     n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                     model_name="censurada", on_error="skip")
    final_clean = sarima_forecast(clean, FORECAST_HORIZON)["yhat"].sum()
    final_raw = sarima_forecast(raw, FORECAST_HORIZON)["yhat"].sum()
    censoring_rows.append({
        "serie": name,
        "mase_censurada": r_raw.metrics["mase"], "mase_corregida": r_clean.metrics["mase"],
        "mape_censurada": r_raw.metrics["mape"], "mape_corregida": r_clean.metrics["mape"],
        "mejora_mase_pct": (1 - r_clean.metrics["mase"] / r_raw.metrics["mase"]) * 100,
        "forecast_6m_censurado": final_raw, "forecast_6m_corregido": final_clean,
        "dif_forecast_pct": (final_clean / final_raw - 1) * 100,
    })
censoring_impact = pd.DataFrame(censoring_rows)
print(censoring_impact.round(3).to_string(index=False))

           serie  mase_censurada  mase_corregida  mape_censurada  mape_corregida  mejora_mase_pct  forecast_6m_censurado  forecast_6m_corregido  dif_forecast_pct
   decaf_vanilla           0.661           0.522          19.576          15.677           21.121              21141.309              23155.111             9.525
      intense_10           0.305           0.305           9.632           9.632            0.000              18006.642              20767.209            15.331
 origin_colombia           0.437           0.399          13.457          12.405            8.661              28366.872              26747.073            -5.710
seasonal_pumpkin           0.306           0.310          40.001          40.529           -1.123              32669.966              33516.536             2.591
           TOTAL           0.374           0.351          12.934          12.086            6.221             209906.712             211816.669             0.910


La comparación se hace con el mismo modelo (`sarima_airline` sobre cada serie) para aislar el efecto
del dato y no mezclarlo con el del modelo. Dos lecturas, y la segunda es la que importa.

**En el backtesting**, corregir la censura reduce el error donde hay ventanas dentro del periodo
evaluado: −21% de MASE en Descafeinado vainilla, −8,7% en Colombia, −6,2% en el total. Intenso 10
sale empatado, y la razón es instructiva: su ventana es de abril-mayo de 2026 y con cinco pliegues a
seis meses el último entrenamiento termina en enero de 2026, así que **ninguna ventana de
entrenamiento llega a contener esa rotura**. El backtesting no puede medir un daño que cae fuera de
su alcance.

**En la previsión que se publica sí lo mide**, y ahí está el golpe: la previsión a seis meses de
Intenso 10 es un **15,3% más alta** con la serie corregida. Es el sabor con la rotura más grande y
más reciente, y con la serie sin corregir el modelo arranca desde un mayo de 2026 artificialmente
hundido y proyecta ese hundimiento hacia adelante.

Es la misma lección que dejó la página de ingresos con el vertido de eventos del último día: **el
origen del forecast es el punto del histórico donde un error de datos hace más daño**, y es también
el punto que ningún backtesting con pliegues hacia atrás llega a auditar.

## 7. Forecast a 6 meses

Se publica el **middle-out**: el mejor error por sabor de los cuatro modelos, cobertura de los doce
SKUs y coherencia exacta entre las partes y el total.

Ninguna de sus dos piezas produce un intervalo utilizable —el reparto por cuota no es un modelo con
distribución propia— así que la incertidumbre tiene que salir de los errores walk-forward. Las dos
páginas anteriores usaban `ts.empirical_interval` para eso: corregir el punto por el sesgo medio de
cada horizonte y suavizar la dispersión con sd(h) = a·√h.

**Esa técnica no se puede trasplantar aquí, y merece la pena enseñar por qué.**

In [20]:
future_months = future_months_of(total_demand.index[-1], FORECAST_HORIZON)
published_parts = middle_out_parts(total_demand.index[-1], FORECAST_HORIZON)
horizons = np.arange(1, FORECAST_HORIZON + 1)

# Diagnóstico: qué factor de corrección pediría la técnica de las páginas 2 y 3 en cada serie.
diagnosis = []
for name, result in [*((f, flavor_backtests[f]["middle_out_grupos"]) for f in FLAVORS),
                     ("TOTAL", total_backtests["middle_out_grupos"])]:
    band = ts.empirical_interval(result.predictions, level=INTERVAL_LEVEL)
    rel = (result.predictions.y_pred - result.predictions.y_true) / result.predictions.y_true
    diagnosis.append({"serie": name, "n_errores": len(rel), "sesgo_medio_pct": rel.mean() * 100,
                      "factor_h1": band.calibration_factor.iloc[0],
                      "factor_h6": band.calibration_factor.iloc[-1]})
diagnosis = pd.DataFrame(diagnosis)
print("Factor de corrección por sesgo que pediría ts.empirical_interval:")
print(diagnosis.round(3).to_string(index=False))

Factor de corrección por sesgo que pediría ts.empirical_interval:
               serie  n_errores  sesgo_medio_pct  factor_h1  factor_h6
    classic_espresso         30            7.770      0.957      0.882
       classic_lungo         30            2.507      0.988      0.917
       decaf_classic         30           -2.322      1.058      0.983
       decaf_vanilla         30            8.784      0.930      0.941
          intense_10         30            3.020      1.013      0.926
          intense_12         30           11.481      0.899      0.924
           intense_8         30            3.168      0.926      1.000
       origin_brazil         30           10.296      1.004      0.833
     origin_colombia         30            5.249      1.033      0.916
     origin_ethiopia         30           13.417      0.973      0.859
seasonal_gingerbread         18          -53.369      1.961      2.122
    seasonal_pumpkin         30          -27.116      2.224      0.995
           

En las diez series de base y en el total el factor de corrección se mueve entre 0,83 y 1,07: un
ajuste fino sobre un sesgo real y pequeño, que es para lo que se diseñó. En las dos de temporada pide
**multiplicar por más de dos**: 2,22 en Calabaza a un mes vista, y entre 1,96 y 2,12 en Jengibre a lo
largo de todo el horizonte.

Ese factor no mide un sesgo, es un artefacto de mezclar niveles incomparables. El sesgo relativo medio
sólo significa algo cuando los puntos que se promedian están en una escala parecida; en un SKU cuya
demanda oscila doce a uno, un error de 200 unidades en octubre y otro de 200 en enero dan errores
relativos que se diferencian en un orden de magnitud, y la media queda dominada por los meses de
valle. Aplicar ese factor subiría la previsión de Calabaza para septiembre de 7.500 a 16.700
unidades, casi seis veces lo que vendió el septiembre anterior.

Hay además un problema propio de Jengibre que ningún reescalado arregla: **su backtesting evalúa un
modelo peor del que se publica**. Cada pliegue entrena con 13 a 15 meses de histórico, mientras que
la previsión publicada tiene 21. El ×2,1 que pide es el sesgo del modelo de hace un año, no el del
que se está usando.

Así que el punto publicado es el del modelo, **sin corregir** —lo que además mantiene intacta la
coherencia entre las partes y el total— y el trabajo de absorber ese sesgo se le pasa a la banda: la
dispersión se mide **sin centrar** (error cuadrático medio del logaritmo del cociente, no desviación
típica alrededor de la media) y en escala logarítmica, que es la escala en la que está modelada la
serie y la única en la que una banda multiplicativa no puede bajar de cero.

In [21]:
from scipy.stats import norm

def log_band(result, level=INTERVAL_LEVEL):
    """
    Banda multiplicativa a partir de los errores walk-forward, en escala logarítmica.

    Dos diferencias con ts.empirical_interval, las dos deliberadas:
      - no se centra: s(h) sale del error cuadrático medio de log(y_pred / y_true) y no de su
        desviación típica, así que un modelo sesgado se paga con una banda más ancha en vez de
        con un reescalado del punto que en las series estacionales no es estimable;
      - se trabaja en logaritmos, que es como está modelada la serie y evita bandas negativas.

    La forma s(h) = a·raíz(h) es la misma de las páginas 2 y 3: agrupa los errores de todos los
    horizontes en vez de estimar seis desviaciones típicas con cinco puntos cada una.
    """
    pred = result.predictions
    log_error = np.log(pred.y_pred.to_numpy() / pred.y_true.to_numpy())
    a_squared = float(np.mean(log_error ** 2 / pred.h.to_numpy()))
    z = float(norm.ppf(0.5 + level / 2))
    sigma = np.sqrt(a_squared * horizons)
    return pd.DataFrame({"s_log": sigma,
                         "lower_factor": np.exp(-z * sigma),
                         "upper_factor": np.exp(z * sigma)}, index=horizons)

def with_band(point, result):
    band = log_band(result)
    return band, pd.DataFrame({"yhat": point,
                               "yhat_lower": point * band.lower_factor.to_numpy(),
                               "yhat_upper": point * band.upper_factor.to_numpy()},
                              index=future_months)

bands, flavor_forecast = {}, {}
for flavor in FLAVORS:
    bands[flavor], flavor_forecast[flavor] = with_band(
        published_parts[flavor], flavor_backtests[flavor]["middle_out_grupos"])
total_band, final_total = with_band(sum(published_parts.values()),
                                    total_backtests["middle_out_grupos"])
sarima_total = sarima_forecast(total_demand, FORECAST_HORIZON)

width = pd.DataFrame({**{FLAVOR_LABEL[f]: bands[f].upper_factor / bands[f].lower_factor
                         for f in FLAVORS},
                      "TOTAL": total_band.upper_factor / total_band.lower_factor})
print("Anchura de la banda 80%, como cociente extremo superior / extremo inferior:")
print(width.T.round(2).to_string())
print()
print("¿las partes suman el total?",
      bool(np.allclose(sum(f.yhat for f in flavor_forecast.values()).to_numpy(),
                       final_total.yhat.to_numpy())))
print()
summary_fc = pd.DataFrame({FLAVOR_LABEL[f]: flavor_forecast[f].yhat for f in FLAVORS},
                          index=future_months.strftime("%Y-%m"))
print("Previsión de unidades por sabor:")
print(summary_fc.round(0).T.to_string())
print()
print(f"total 6 meses: {final_total.yhat.sum():,.0f} unidades "
      f"[{final_total.yhat_lower.sum():,.0f} – {final_total.yhat_upper.sum():,.0f}]")

fig = go.Figure()
fig.add_trace(go.Scatter(x=total_demand.index, y=total_demand.to_numpy(), name="Histórico",
                         mode="lines", line=dict(color=C_INK, width=2.5)))
fig.add_trace(go.Scatter(x=[*future_months, *future_months[::-1]],
                         y=[*final_total.yhat_upper, *final_total.yhat_lower[::-1]],
                         fill="toself", fillcolor="rgba(42,120,214,0.14)", line=dict(width=0),
                         name="Banda 80%", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[total_demand.index[-1], *future_months],
                         y=[float(total_demand.iloc[-1]), *final_total.yhat],
                         name="Middle-out (publicado)", mode="lines+markers",
                         line=dict(color=C_BLUE, width=2.5, dash="dash")))
fig.add_trace(go.Scatter(x=[total_demand.index[-1], *future_months],
                         y=[float(total_demand.iloc[-1]), *sarima_total.yhat],
                         name="SARIMA sobre el total", mode="lines",
                         line=dict(color=C_MUTED, width=1.8, dash="dot")))
fig.update_layout(**{**PLOT_LAYOUT, "height": 440},
                  title="Demanda total de cápsulas: histórico y previsión a 6 meses",
                  yaxis_title="unidades / mes")
fig.show()

SHOWCASE = ("classic_espresso", "origin_colombia", "seasonal_pumpkin", "seasonal_gingerbread")
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.14, horizontal_spacing=0.08,
                    subplot_titles=[FLAVOR_LABEL[f] for f in SHOWCASE])
for k, flavor in enumerate(SHOWCASE):
    r, c = divmod(k, 2)
    fc = flavor_forecast[flavor]
    hist = monthly[flavor][monthly[flavor] > 0]
    fig.add_trace(go.Scatter(x=hist.index, y=hist.to_numpy(), mode="lines",
                             line=dict(color=C_INK, width=2), showlegend=False), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=[*fc.index, *fc.index[::-1]],
                             y=[*fc.yhat_upper, *fc.yhat_lower[::-1]], fill="toself",
                             fillcolor="rgba(42,120,214,0.14)", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=[hist.index[-1], *fc.index], y=[float(hist.iloc[-1]), *fc.yhat],
                             mode="lines", line=dict(color=FLAVOR_COLOR[flavor], width=2.2,
                                                     dash="dash"),
                             showlegend=False), row=r + 1, col=c + 1)
fig.update_layout(**{**PLOT_LAYOUT, "height": 560, "hovermode": "closest"},
                  title="Previsión por sabor: dos de base y dos de temporada (banda 80%)")
fig.show()

Anchura de la banda 80%, como cociente extremo superior / extremo inferior:
                       1     2     3      4      5      6
Espresso clásico    1.24  1.36  1.45   1.54   1.62   1.70
Lungo clásico       1.22  1.32  1.41   1.49   1.56   1.63
Descafeinado        1.22  1.32  1.41   1.49   1.56   1.63
Descaf. vainilla    1.25  1.38  1.48   1.57   1.66   1.74
Intenso 10          1.21  1.32  1.40   1.47   1.54   1.61
Intenso 12          1.31  1.46  1.59   1.71   1.82   1.93
Intenso 8           1.28  1.42  1.54   1.65   1.75   1.84
Origen Brasil       1.20  1.29  1.37   1.44   1.50   1.56
Origen Colombia     1.21  1.31  1.39   1.46   1.53   1.59
Origen Etiopía      1.33  1.50  1.65   1.78   1.90   2.03
Temporada jengibre  3.37  5.58  8.22  11.39  15.17  19.67
Temporada calabaza  3.45  5.76  8.54  11.90  15.94  20.76
TOTAL               1.19  1.28  1.35   1.42   1.48   1.54

¿las partes suman el total? True

Previsión de unidades por sabor:
                    2026-09  2026-10  2026-1

La banda separa el catálogo en dos con una claridad que ningún otro gráfico de esta página consigue.
A seis meses, en el total el extremo superior es **1,5 veces el inferior**, y en los diez sabores de
base entre 1,6 y 2,0. En Calabaza y Jengibre es **de veinte a uno**.

No es un defecto del modelo, es la medida honesta de lo que se sabe: Jengibre tiene una sola Navidad
completa en el histórico, así que su previsión de diciembre descansa sobre una observación. Una banda
estrecha ahí sería una mentira cómoda, y es exactamente la mentira que habría producido la
calibración de las páginas anteriores: al centrar la dispersión, un sesgo grande y **estable** —como
el de Jengibre, que se equivoca por un factor dos en casi todos los pliegues— se convierte en una
desviación típica pequeña y por tanto en una banda estrechísima alrededor de un punto mal escalado.
Medir sin centrar es lo que impide ese resultado.

**Lectura de compras**, que es para lo que sirve esta página. Los diez sabores de base se pueden
comprar contra la previsión con un margen razonable, y ahí es donde estaban las seis roturas
detectadas: 17.119 € de demanda perdida sobre los SKUs **más previsibles del catálogo**, que con la
serie corregida y el extremo superior de la banda como objetivo de cobertura es un coste evitable.
Los dos de temporada son el problema contrario —el pico de Calabaza en octubre y el de Jengibre en
diciembre se conocen con seis meses de antelación, pero con un factor de error de veinte— y ahí la
respuesta no es un modelo mejor sino una decisión de negocio sobre cuánto sobrestock se está
dispuesto a asumir en dos meses concretos del año.

## 8. Volcado a `analysis/outputs/capsulas.json`

In [22]:
def records(series_obj, key="value"):
    return ts.series_to_records(series_obj, value_key=key)

# Serie diaria sólo alrededor de las ventanas detectadas: es lo que necesita el gráfico de la
# censura, sin arrastrar 12 x 1.096 puntos al JSON del informe.
censorship_detail = []
for row in stockouts.itertuples():
    lo, hi = row.start - pd.Timedelta(days=60), row.end + pd.Timedelta(days=60)
    window = (CALENDAR >= lo) & (CALENDAR <= hi)
    censorship_detail.append({
        "flavor": row.flavor, "label": FLAVOR_LABEL[row.flavor],
        "start": row.start.strftime("%Y-%m-%d"), "end": row.end.strftime("%Y-%m-%d"),
        "days": int(row.days),
        "shop_lines": records(shop_lines[row.flavor][window], "lines"),
        "shop_units_observed": records(shop_units[row.flavor][window], "units"),
        "shop_units_imputed": records(shop_units_adj[row.flavor][window], "units"),
        "club_units": records(club_units[row.flavor][window], "units"),
    })

base_profile_published = share_profile(monthly[BASE_FLAVORS], total_demand.index[-1])

payload = {
    "meta": {
        "page": "04_capsulas",
        "title": "Serie temporal: demanda por sabor de cápsula",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_tables": ["fct_shipments", "fct_shop_orders", "int_product_sku_continuity"],
        "grain": "mensual por sabor (canonical_sku); diario en el detalle de la censura",
        "history_start": total_demand.index.min().strftime("%Y-%m-%d"),
        "history_end": total_demand.index.max().strftime("%Y-%m-%d"),
        "n_months": int(len(total_demand)),
        "n_flavors": len(FLAVORS),
        "forecast_horizon": FORECAST_HORIZON,
        "interval_level": INTERVAL_LEVEL,
        "measure": "unidades (cápsulas) = fct_shipments.quantity + fct_shop_orders.quantity",
        "measure_note": (
            "La demanda se mide en unidades y no en euros a propósito: el relanzamiento de tres "
            "SKUs sube el precio entre un 9% y un 11%, así que la serie en euros tiene un escalón "
            "real de precio que se confundiría con un cambio de demanda."),
        "channel_split_pct": {
            "tienda": float(shop.units.sum() / (shop.units.sum() + club.units.sum()) * 100),
            "club": float(club.units.sum() / (shop.units.sum() + club.units.sum()) * 100)},
        "total_units": float(total_demand.sum()),
    },
    "flavors": [{"flavor": f, "label": FLAVOR_LABEL[f], "color": FLAVOR_COLOR[f],
                 "is_seasonal": f in SEASONAL_FLAVORS,
                 "was_relaunched": bool(f in switch_dates),
                 "share_pct": float(share_total[f]),
                 "share_last_12m_pct": float(share_recent[f]),
                 "months_of_history": int((monthly[f] > 0).sum())}
                for f in FLAVORS],
    "series": {
        "total": records(total_demand, "units"),
        "by_flavor": {f: records(monthly[f].asfreq("MS"), "units") for f in FLAVORS},
        "by_flavor_censored": {f: records(monthly_raw[f].asfreq("MS"), "units")
                               for f in sorted(stockouts.flavor.unique())},
        "shop_daily": records(shop_daily_total, "units"),
        "club_daily": records(club_daily_total, "units"),
    },
    "sku_relaunch": {
        "imperfection": "SKUs descatalogados y relanzados con código nuevo",
        "n_flavors": len(switch_dates),
        "treatment": ("Se agrega por canonical_sku, que resuelve la cadena replaced_by_sku en "
                      "int_product_sku_continuity. En unidades la discontinuidad es un artefacto "
                      "de codificación; en euros el escalón de precio es real y no se corrige."),
        "price_change": json.loads(price_change.astype({"fecha_cambio": str}).to_json(orient="records")),
        "sku_level_series": [
            {"flavor": f, "label": FLAVOR_LABEL[f],
             "switch_date": switch_dates[f].strftime("%Y-%m-%d"),
             "by_sku": {sku: [{"date": d.strftime("%Y-%m-%d"), "units": float(u)}
                              for d, u in zip(part.d, part.units)]
                        for sku, part in sku_monthly[sku_monthly.flavor == f].groupby("product_sku")}}
            for f in switch_dates],
        "volume_effect": {
            "method": ("Diferencias en diferencias sobre unidades, control = sabores no relanzados "
                       "y no estacionales, ventana de 120 días. El IC de Poisson se muestra sólo "
                       "para enseñar que engaña: la demanda diaria está sobredispersa 11-18 veces."),
            "overdispersion_ratio": {f: float(v) for f, v in overdispersion.items()},
            "estimates": json.loads(did_table.to_json(orient="records")),
            "verdict": ("Ningún relanzamiento mueve el volumen de forma medible. En jengibre el "
                        "efecto no es identificable: coincide con el final de su temporada."),
        },
    },
    "censored_demand": {
        "imperfection": "Roturas de stock que censuran la demanda de tienda",
        "detection": {
            "method": ("Contraste de Poisson sobre cada racha de ceros dentro de la vida de "
                       "catálogo: la tasa esperada se estima con la cuota local del sabor sobre "
                       "el resto del catálogo (+-45 días), y se contrasta P(0 ventas en toda la "
                       "ventana). Bonferroni sobre las rachas contrastadas, más un umbral de "
                       "materialidad de 20 líneas."),
            "control": ("El club no se censura, así que sirve de control: durante las seis "
                        "ventanas siguió sirviendo el sabor con normalidad mientras la tienda "
                        "marcaba cero."),
            "n_runs_tested": int(len(candidates)),
            "n_significant": int(candidates.significant.sum()),
            "n_detected": int(len(stockouts)),
            "min_run_days": MIN_RUN_DAYS, "ref_days": REF_DAYS,
            "min_expected_lines": MIN_EXPECTED_LINES, "alpha": STOCKOUT_ALPHA,
        },
        "candidates": json.loads(
            candidates.assign(start=candidates.start.dt.strftime("%Y-%m-%d"),
                              end=candidates.end.dt.strftime("%Y-%m-%d")).to_json(orient="records")),
        "windows": json.loads(
            stockouts.assign(start=stockouts.start.dt.strftime("%Y-%m-%d"),
                             end=stockouts.end.dt.strftime("%Y-%m-%d")).to_json(orient="records")),
        "validation": {
            "note": ("El manifiesto del generador no se carga en DuckDB ni se usa para detectar; "
                     "se abre después para medir el acierto del procedimiento."),
            "n_real": int(len(truth)), "n_detected": int(len(stockouts)),
            "true_positives": len(found & real), "false_positives": len(found - real),
            "false_negatives": len(real - found),
            "real_lines": int(check.lineas_reales.sum()),
            "estimated_lines": float(check.lineas_estimadas.sum()),
            "aggregate_error_pct": float(check.lineas_estimadas.sum() / check.lineas_reales.sum() - 1) * 100,
            "by_window": json.loads(check.assign(start=check.start.dt.strftime("%Y-%m-%d"))
                                    [["flavor", "start", "end", "lineas_reales",
                                      "lineas_estimadas", "error_pct"]].to_json(orient="records")),
        },
        "lost_demand": {
            "units": float(imputed_units.sum()),
            "eur": float((imputed_units * avg_price).sum()),
            "pct_of_shop_capsule_revenue": float(
                (imputed_units * avg_price).sum() / shop_eur.to_numpy().sum() * 100),
            "by_flavor": json.loads(lost.reset_index().to_json(orient="records")),
        },
        "monthly_impact": [
            {"date": d.strftime("%Y-%m-%d"), "censored": float(impact.censurada[d]),
             "corrected": float(impact.corregida[d]), "diff_pct": float(impact.dif_pct[d])}
            for d in impact.index if impact.dif_pct[d] > 0.05
        ],
        "daily_detail": censorship_detail,
    },
    "decomposition": decomposition.to_dict(),
    "flavor_decomposition_strength": json.loads(strength.to_json(orient="records")),
    "seasonal_profile": {
        "note": ("Índice demanda / tendencia, mediana de los últimos 24 meses. 1,00 = mes en línea "
                 "con la tendencia."),
        "regimes": {
            "base": {"flavors": BASE_FLAVORS,
                     "peak_month": int(base_profile.idxmax()),
                     "trough_month": int(base_profile.idxmin()),
                     "amplitude_pct": float((base_profile.max() / base_profile.min() - 1) * 100)},
            "seasonal": {f: {"peak_month": int(profile_frame[f].idxmax()),
                             "trough_month": int(profile_frame[f].idxmin()),
                             "amplitude_pct": float((profile_frame[f].max()
                                                     / profile_frame[f].min() - 1) * 100)}
                         for f in SEASONAL_FLAVORS},
        },
        "by_month": [{"month": int(m), "label": MONTHS[int(m) - 1],
                      **{f: (float(profile_frame.loc[m, f])
                             if pd.notna(profile_frame.loc[m, f]) else None) for f in FLAVORS}}
                     for m in range(1, 13)],
    },
    "weekly_seasonality": {
        "note": ("El club no tiene ciclo semanal (los envíos los programa el almacén) y la tienda "
                 "sí. Es el 31% de la demanda que se puede reprogramar."),
        "by_channel": {
            name: {"seasonal_strength_7": float(w["decomposition"].seasonal_strength[7]),
                   "seasonal_strength_365": (float(w["decomposition"].seasonal_strength[365])
                                             if 365 in w["decomposition"].seasonal_strength else None),
                   "trend_strength": float(w["decomposition"].trend_strength),
                   "profile_pct": [{"weekday": i, "label": DAYS[i], "pct": float(w["profile"].iloc[i])}
                                   for i in range(7)]}
            for name, w in weekly.items()},
    },
    "stationarity": {"level": report_level.to_dict(), "log": report_log.to_dict(),
                     "by_flavor": {f: r.to_dict() for f, r in flavor_stationarity.items()}},
    "backtest": {
        "horizon": FORECAST_HORIZON, "n_folds": BACKTEST_FOLDS, "window": "expanding",
        "total_leaderboard": json.loads(leaderboard.to_json(orient="records")),
        "total_models": {name: r.to_dict() for name, r in total_backtests.items()},
        "per_flavor": json.loads(per_flavor.to_json(orient="records")),
        "per_flavor_mean_mase": {c.replace("mase_", ""): float(per_flavor[c].mean())
                                 for c in MASE_COLS},
        "hierarchy_note": ("En los sabores el top-down y el bottom-up empatan; en el total gana el "
                           "bottom-up, al revés que en la página de ingresos, porque el agregado "
                           "mezcla dos calendarios estacionales incompatibles. El middle-out —los "
                           "diez sabores de base como un bloque, los dos de temporada aparte— gana "
                           "en diez de los doce sabores y queda segundo en el total."),
        "censoring_impact": json.loads(censoring_impact.to_json(orient="records")),
        "censoring_note": ("Mismo objetivo de evaluación —la serie corregida— y dos entrenamientos: "
                           "con la serie del mart y con la serie corregida, siempre con el mismo "
                           "modelo. Intenso 10 empata porque su rotura cae fuera de toda ventana de "
                           "entrenamiento del backtesting, pero sí mueve la previsión un +15,3%."),
    },
    "forecast": {
        "horizon": FORECAST_HORIZON,
        "months": [d.strftime("%Y-%m-%d") for d in future_months],
        "published_model": "middle_out_grupos",
        "published_model_note": ("Middle-out: los diez sabores de base se pronostican como un "
                                 "bloque y se reparten por su cuota dentro del bloque; los dos de "
                                 "temporada llevan modelo propio. Mejor MASE medio por sabor de "
                                 "los cuatro modelos (0,384) y mejor en siete de los doce, cubre "
                                 "los doce SKUs y las partes suman el total por construcción."),
        "total": {"label": "Middle-out (suma coherente de los doce sabores)",
                  "yhat": [float(v) for v in final_total.yhat],
                  "yhat_lower": [float(v) for v in final_total.yhat_lower],
                  "yhat_upper": [float(v) for v in final_total.yhat_upper],
                  "interval_source": "empírico sin centrar, en escala log, de los errores "
                                     "walk-forward"},
        "total_sarima_reference": {
            "label": "SARIMA(0,1,1)(0,1,1)12 sobre el log del total, como referencia",
            "yhat": [float(v) for v in sarima_total.yhat],
            "yhat_lower": [float(v) for v in sarima_total.yhat_lower],
            "yhat_upper": [float(v) for v in sarima_total.yhat_upper],
            "interval_source": "analítico del modelo"},
        "base_block_share_profile_pct": [
            {"month": int(m), "label": MONTHS[int(m) - 1],
             **{f: float(base_profile_published.loc[m, f] * 100) for f in BASE_FLAVORS}}
            for m in base_profile_published.index],
        "interval_method": {
            "note": ("El punto no se corrige por sesgo. La técnica de las páginas 2 y 3 "
                     "(ts.empirical_interval) pide multiplicar por 2,2 las dos series de "
                     "temporada, porque el sesgo relativo medio no es estimable cuando se "
                     "promedian meses cuyo nivel se diferencia en un orden de magnitud. La banda "
                     "mide la dispersión SIN centrar, sobre log(y_pred/y_true), con s(h)=a*raíz(h)."),
            "bias_diagnosis": json.loads(diagnosis.to_json(orient="records")),
            "band_width_ratio": [
                {"h": int(h),
                 **{f: float(bands[f].upper_factor[h] / bands[f].lower_factor[h]) for f in FLAVORS},
                 "TOTAL": float(total_band.upper_factor[h] / total_band.lower_factor[h])}
                for h in horizons],
        },
        "by_flavor": {
            f: {"label": FLAVOR_LABEL[f],
                "group": "base" if f in BASE_FLAVORS else "temporada",
                "yhat": [float(v) for v in flavor_forecast[f].yhat],
                "yhat_lower": [float(v) for v in flavor_forecast[f].yhat_lower],
                "yhat_upper": [float(v) for v in flavor_forecast[f].yhat_upper],
                "interval_source": "empírico sin centrar, en escala log, del middle-out"}
            for f in FLAVORS},
    },
    "insights": [
        ("Las seis roturas de stock se detectan sin mirar el ground truth: contraste de Poisson "
         "sobre la tasa local esperada, Bonferroni y un umbral de materialidad de 20 líneas. "
         "6 aciertos, 0 falsos positivos, y la demanda censurada estimada se desvía un +0,7% de "
         "la real (972 líneas frente a 966)."),
        ("La clave para distinguir una rotura de stock de una caída de demanda es que el club no "
         "se censura: durante las seis ventanas siguió sirviendo el sabor con normalidad mientras "
         "la tienda marcaba cero."),
        ("Ordenar las rachas de ceros por duración no sirve: las más largas del histórico son de "
         "sabores que en 2023 vendían 0,2 líneas al día. Lo que hay que medir es cuánta demanda "
         "debería haber habido dentro de la ventana, no cuántos días dura."),
        ("La censura vale 3.594 unidades y 17.119 € (1,35% de la facturación de cápsulas en "
         "tienda), pero su daño no es proporcional a su tamaño: tres de las seis ventanas caen en "
         "los últimos siete meses del histórico, que es el origen del forecast."),
        ("Tratar la censura mejora el backtesting un 21% de MASE en descafeinado vainilla y un "
         "6,2% en el total, y sube la previsión a 6 meses de Intenso 10 un 15,3%. El backtesting "
         "no llega a ver esa rotura porque cae fuera de sus ventanas de entrenamiento."),
        ("El relanzamiento de SKU no movió el volumen de ningún sabor. El -8,8% aparente de Brasil "
         "es ruido: la demanda diaria está sobredispersa 11-18 veces, y un placebo sobre 42 fechas "
         "cualesquiera da una desviación típica del 11,9%."),
        ("Hay dos regímenes de demanda: diez sabores de base con un mismo calendario (techo en "
         "marzo, suelo en agosto, amplitud del 47%) y dos de temporada con amplitudes de 12 a 1 y "
         "de 27 a 1. Son el 9% del volumen y en el agregado son invisibles."),
        ("El nivel al que conviene modelar lo fija la homogeneidad estacional, no la jerarquía del "
         "catálogo: agrupar los diez sabores de base y dejar aparte los dos de temporada "
         "(middle-out) gana en siete de los doce sabores, con un MASE medio de 0,384 frente a "
         "0,450 del top-down y 0,478 del bottom-up."),
        ("La calibración empírica de las páginas 2 y 3 no se puede trasplantar a un SKU con doce a "
         "uno de amplitud estacional: pide multiplicar por 2,2 la previsión de Calabaza. El punto "
         "se publica sin corregir y la banda se mide sin centrar y en logaritmos, lo que deja a "
         "los dos sabores de temporada con una banda de veinte a uno frente al 1,5 a 1 del "
         "total: es la medida honesta de lo poco que se sabe de ellos."),
        ("El club no tiene estacionalidad semanal (F_S = 0,22, perfil plano a ±4%) y la tienda sí "
         "(F_S = 0,63, domingo -29%). El 31% de la demanda que va por el club es la parte "
         "reprogramable del pico semanal."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUTPUT_PATH.stat().st_size / 1024:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\capsulas.json (692 KB)
Claves de primer nivel: ['meta', 'flavors', 'series', 'sku_relaunch', 'censored_demand', 'decomposition', 'flavor_decomposition_strength', 'seasonal_profile', 'weekly_seasonality', 'stationarity', 'backtest', 'forecast', 'insights']


In [23]:
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "serie total": len(reloaded["series"]["total"]) == len(total_demand),
    "doce sabores": len(reloaded["series"]["by_flavor"]) == len(FLAVORS) == 12,
    "series por sabor completas": all(len(v) == len(total_demand)
                                      for v in reloaded["series"]["by_flavor"].values()),
    "series diarias": (len(reloaded["series"]["shop_daily"]) == len(CALENDAR)
                       and len(reloaded["series"]["club_daily"]) == len(CALENDAR)),
    "seis roturas detectadas": len(reloaded["censored_demand"]["windows"]) == 6,
    "validación 6/6 sin falsos positivos": (
        reloaded["censored_demand"]["validation"]["true_positives"] == 6
        and reloaded["censored_demand"]["validation"]["false_positives"] == 0),
    "detalle diario de cada ventana": len(reloaded["censored_demand"]["daily_detail"]) == 6,
    "tres relanzamientos": len(reloaded["sku_relaunch"]["price_change"]) == 3,
    "efecto de volumen estimado": len(reloaded["sku_relaunch"]["volume_effect"]["estimates"]) == 3,
    "descomposición": len(reloaded["decomposition"]["trend"]) == len(total_demand),
    "perfil estacional de 12 meses": len(reloaded["seasonal_profile"]["by_month"]) == 12,
    "estacionalidad semanal por canal": all(
        len(v["profile_pct"]) == 7 for v in reloaded["weekly_seasonality"]["by_channel"].values()),
    "leaderboard del total": len(reloaded["backtest"]["total_leaderboard"]) == len(total_models) == 4,
    "backtest por sabor": len(reloaded["backtest"]["per_flavor"]) == len(FLAVORS),
    "impacto de la censura": len(reloaded["backtest"]["censoring_impact"]) == 5,
    "forecast del total": (len(reloaded["forecast"]["total"]["yhat"]) == FORECAST_HORIZON
                           and len(reloaded["forecast"]["total"]["yhat_lower"]) == FORECAST_HORIZON),
    "forecast por sabor con banda": all(
        len(v["yhat"]) == FORECAST_HORIZON and len(v["yhat_lower"]) == FORECAST_HORIZON
        for v in reloaded["forecast"]["by_flavor"].values()),
    "forecast coherente": np.allclose(
        sum(np.asarray(v["yhat"]) for v in reloaded["forecast"]["by_flavor"].values()),
        np.asarray(reloaded["forecast"]["total"]["yhat"])),
    "diagnóstico de sesgo de las 13 series": (
        len(reloaded["forecast"]["interval_method"]["bias_diagnosis"]) == len(FLAVORS) + 1),
    "previsión positiva": all(min(v["yhat_lower"]) > 0
                              for v in reloaded["forecast"]["by_flavor"].values()),
    "bandas ordenadas": all(
        np.all(np.asarray(v["yhat_lower"]) <= np.asarray(v["yhat"]))
        and np.all(np.asarray(v["yhat"]) <= np.asarray(v["yhat_upper"]))
        for v in reloaded["forecast"]["by_flavor"].values()),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print()
print("JSON verificado.")

  OK  serie total
  OK  doce sabores
  OK  series por sabor completas
  OK  series diarias
  OK  seis roturas detectadas
  OK  validación 6/6 sin falsos positivos
  OK  detalle diario de cada ventana
  OK  tres relanzamientos
  OK  efecto de volumen estimado
  OK  descomposición
  OK  perfil estacional de 12 meses
  OK  estacionalidad semanal por canal
  OK  leaderboard del total
  OK  backtest por sabor
  OK  impacto de la censura
  OK  forecast del total
  OK  forecast por sabor con banda
  OK  forecast coherente
  OK  diagnóstico de sesgo de las 13 series
  OK  previsión positiva
  OK  bandas ordenadas

JSON verificado.


## Conclusiones

1. **Un cero de demanda y un cero de oferta son el mismo número y dos cosas opuestas.** Las seis
   roturas de stock se encuentran sin mirar el ground truth, contrastando cuánta demanda debería
   haber habido en cada racha de ceros en vez de cuántos días dura. Seis aciertos, ningún falso
   positivo, y una estimación del tamaño de la censura que se desvía un +0,7% de la real.
2. **La imperfección de un canal se detecta con el canal que no la tiene.** Que la rotura censure la
   tienda pero no el club convierte al club en grupo de control, y es lo que separa "no había
   producto" de "nadie lo quería". Es la misma idea que en la página de ingresos —buscar un
   denominador que no se mueva por la misma causa— aplicada a un canal en vez de a una métrica.
3. **Corregir la censura importa más en la previsión que en el backtesting.** El backtesting mejora
   un 6,2% de MASE en el total, pero la previsión a seis meses de Intenso 10 sube un 15,3%: su
   rotura es la más grande y la más reciente, y cae justo en el origen del forecast, que es el punto
   que ningún pliegue hacia atrás llega a auditar.
4. **Un salto en la serie no siempre es un salto en el negocio.** El relanzamiento de tres SKUs rompe
   la serie en unidades por un cambio de código —se arregla con `canonical_sku`— y la rompe en euros
   por una subida real de precio del 9-11%, que no hay que tocar. Y el efecto sobre el volumen es
   nulo: lo que parecía un −8,8% significativo en Brasil se deshace en cuanto se reconoce que la
   demanda diaria está sobredispersa 11-18 veces y se contrasta contra un placebo.
5. **El agregado no es un resumen de las partes.** Los dos sabores de temporada son el 9% del volumen
   y tienen amplitudes estacionales de 12 a 1 y de 27 a 1; en la serie total no se ven, y son justo
   los que deciden el calendario de compras. Ésa es la razón de ser de una página a nivel de SKU.
6. **El nivel al que modelar lo fija la estacionalidad, no la jerarquía.** Ni el bottom-up puro ni el
   top-down puro aciertan: el primero deja sin modelo al SKU de 21 meses, y el segundo obliga a un
   único SARIMA a ajustar la mezcla de dos calendarios. Agrupar los diez sabores que comparten ciclo
   y dejar aparte los dos que no —middle-out— gana en siete de los doce sabores y un 15-20% de MASE
   medio, mantiene la coherencia entre las partes y el total, y cubre el catálogo entero.
7. **Una técnica que funcionó en la página anterior no se hereda sin comprobarla.** La calibración
   empírica de las páginas 2 y 3 pide aquí multiplicar por 2,2 la previsión de Calabaza, porque un
   sesgo relativo medio no significa nada cuando se promedian meses cuyo nivel se diferencia en un
   orden de magnitud. El punto se publica sin corregir y la banda absorbe el sesgo midiéndose sin
   centrar: el resultado es una banda de veinte a uno en los dos sabores de temporada y de 1,5 a 1
   en el total, que es exactamente lo que se sabe de cada uno.